# Explore and visualize the data

Produces the descriptive figures used to assess bidder participation, award values, supplier activity, recurrence, and buyer concentration.

Run this notebook from the `notebooks/` directory after completing the preceding numbered stage. Generated files are written to the documented project directories.


In [ ]:
# ============================================================
# ============================================================
# análisis exploratorio correspondientes al apartado 4.4.
#
# variables.
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

ruta_procesados = "../data/processed"

ruta_figuras = "../outputs/figures"

os.makedirs(ruta_figuras, exist_ok=True)

print("Entorno preparado correctamente.")
print("Ruta de datos:", ruta_procesados)
print("Ruta de figuras:", ruta_figuras)


In [ ]:
# ============================================================
# ============================================================
#
# ============================================================

procedimientos_df = pd.read_csv(
    os.path.join(
        ruta_procesados,
        "procedures_2025.csv"
    ),
    dtype={
        "buyer_ruc": "string",
        "buyer_ruc_valido": "string"
    }
)

oferentes_df = pd.read_csv(
    os.path.join(
        ruta_procesados,
        "tender_participation_2025.csv"
    ),
    dtype={
        "ruc_oferente_validado": "string"
    }
)

adjudicaciones_df = pd.read_csv(
    os.path.join(
        ruta_procesados,
        "awards_2025.csv"
    ),
    dtype={
        "supplier_ruc": "string",
        "supplier_ruc_validado": "string",
        "cpc_id": "string",
        "cpc_5": "string"
    }
)

print("Procedimientos:", procedimientos_df.shape)
print("Analyze bidder participation:", oferentes_df.shape)
print("Adjudicaciones:", adjudicaciones_df.shape)


In [ ]:
print("=" * 70)
print("CONTROL DE ESTRUCTURA - ADJUDICACIONES")
print("=" * 70)

print("\nNúmero total de filas:")
print(f"{len(adjudicaciones_df):,}")

print("\nOCID únicos:")
print(f"{adjudicaciones_df['ocid'].nunique():,}")

print("\nCombinaciones OCID + proveedor únicas:")
print(
    adjudicaciones_df[
        ["ocid", "supplier_id"]
    ].drop_duplicates().shape[0]
)

print("\nAward ID disponible:")
print("award_id" in adjudicaciones_df.columns)

if "award_id" in adjudicaciones_df.columns:
    print("\nCombinaciones OCID + award_id + proveedor únicas:")
    print(
        adjudicaciones_df[
            ["ocid", "award_id", "supplier_id"]
        ].drop_duplicates().shape[0]
    )

    print("\nDuplicados OCID + award_id + proveedor:")
    print(
        adjudicaciones_df.duplicated(
            subset=["ocid", "award_id", "supplier_id"]
        ).sum()
    )

print("\nMáximo número de filas del mismo proveedor dentro de un OCID:")
print(
    adjudicaciones_df
    .groupby(["ocid", "supplier_id"])
    .size()
    .max()
)


In [ ]:
# ============================================================
# ============================================================
#
# iniciados durante 2025.
#
# CORRECCIÓN METODOLÓGICA:
# ============================================================


# ------------------------------------------------------------
# ------------------------------------------------------------

procedimientos_df["fecha_inicio_dt"] = pd.to_datetime(
    procedimientos_df["tender_start_date"],
    errors="coerce"
)


# ------------------------------------------------------------
# ------------------------------------------------------------

adjudicaciones_df["fecha_adjudicacion_dt"] = pd.to_datetime(
    adjudicaciones_df["award_date_local"],
    errors="coerce"
)


# ------------------------------------------------------------
# ------------------------------------------------------------

procedimientos_df["mes_inicio"] = (
    procedimientos_df["fecha_inicio_dt"].dt.month
)


# ------------------------------------------------------------
# ------------------------------------------------------------

nombres_meses = {
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre"
}


procedimientos_df["mes_inicio_nombre"] = (
    procedimientos_df["mes_inicio"]
    .map(nombres_meses)
)


# ============================================================
# ============================================================


# ------------------------------------------------------------
# ------------------------------------------------------------

print("=" * 60)
print("VALIDACIÓN TEMPORAL DEL ESTUDIO")
print("=" * 60)

print("\nFecha mínima de inicio:")

print(
    procedimientos_df[
        "fecha_inicio_dt"
    ].min()
)


# ------------------------------------------------------------
# ------------------------------------------------------------

print("\nFecha máxima de inicio:")

print(
    procedimientos_df[
        "fecha_inicio_dt"
    ].max()
)


# ------------------------------------------------------------
# ------------------------------------------------------------

print("\nFechas de inicio no convertidas:")

print(
    procedimientos_df[
        "fecha_inicio_dt"
    ].isna().sum()
)


# ------------------------------------------------------------
# ------------------------------------------------------------

print("\nFechas de adjudicación no convertidas:")

print(
    adjudicaciones_df[
        "fecha_adjudicacion_dt"
    ].isna().sum()
)


# ============================================================
# ============================================================
# corresponde efectivamente a procedimientos iniciados durante
# 2025.
# ============================================================


# ------------------------------------------------------------
# ------------------------------------------------------------

fuera_2025 = procedimientos_df[
    procedimientos_df[
        "fecha_inicio_dt"
    ].notna()
    &
    (
        procedimientos_df[
            "fecha_inicio_dt"
        ].dt.year != 2025
    )
].copy()


print(
    "\nProcedimientos con fecha de inicio fuera de 2025:"
)

print(
    f"{len(fuera_2025):,}"
)


# ------------------------------------------------------------
# ------------------------------------------------------------

print(
    "\nDistribución por año de inicio:"
)

print(
    procedimientos_df[
        "fecha_inicio_dt"
    ]
    .dt.year
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# Validación final
# ------------------------------------------------------------

if len(fuera_2025) == 0:

    print(
        "\nVALIDACIÓN: Todos los procedimientos con fecha "
        "de inicio válida corresponden al año 2025."
    )

else:

    print(
        "\nADVERTENCIA: Existen procedimientos con fecha "
        "de inicio fuera del año 2025. Revisar antes de "
        "continuar con las visualizaciones."
    )


In [ ]:
# ============================================================
# PASO 5. FIGURA 1
# ============================================================
# Objetivo:
# durante 2025.
#
# CRITERIOS METODOLÓGICOS:
#   "0 oferentes registrados".
# - No mide concentración, recurrencia, riesgo ni irregularidad.
#   mercado correspondiente.
# ============================================================


from IPython.display import display


# ------------------------------------------------------------
# ------------------------------------------------------------

participacion_oferentes = procedimientos_df[
    [
        "ocid",
        "number_of_tenderers"
    ]
].copy()


# ------------------------------------------------------------
# ------------------------------------------------------------

participacion_oferentes["numero_oferentes"] = pd.to_numeric(
    participacion_oferentes["number_of_tenderers"],
    errors="coerce"
)


# ------------------------------------------------------------
# ------------------------------------------------------------

total_procedimientos = len(
    participacion_oferentes
)

faltantes_oferentes = (
    participacion_oferentes[
        "numero_oferentes"
    ]
    .isna()
    .sum()
)

valores_cero = (
    participacion_oferentes[
        "numero_oferentes"
    ]
    .eq(0)
    .sum()
)

valores_negativos = (
    participacion_oferentes[
        "numero_oferentes"
    ]
    .lt(0)
    .sum()
)


print("=" * 65)
print("CONTROL DE PARTICIPACIÓN DE OFERENTES")
print("=" * 65)

print(
    f"\nTotal de procedimientos: "
    f"{total_procedimientos:,}"
)

print(
    f"Valores no informados/no convertibles: "
    f"{faltantes_oferentes:,}"
)

print(
    f"Procedimientos con 0 oferentes registrados: "
    f"{valores_cero:,}"
)

print(
    f"Valores negativos detectados: "
    f"{valores_negativos:,}"
)


# ------------------------------------------------------------
# ------------------------------------------------------------

def clasificar_participacion(numero):

    if pd.isna(numero):
        return "No informado"

    elif numero < 0:
        return "Dato no válido"

    elif numero == 0:
        return "0 oferentes registrados"

    elif numero == 1:
        return "1 oferente"

    elif 2 <= numero <= 3:
        return "2–3 oferentes"

    elif 4 <= numero <= 5:
        return "4–5 oferentes"

    elif 6 <= numero <= 10:
        return "6–10 oferentes"

    else:
        return "Más de 10 oferentes"


participacion_oferentes[
    "categoria_participacion"
] = (
    participacion_oferentes[
        "numero_oferentes"
    ]
    .apply(
        clasificar_participacion
    )
)


# ------------------------------------------------------------
# ------------------------------------------------------------

orden_categorias = [
    "0 oferentes registrados",
    "1 oferente",
    "2–3 oferentes",
    "4–5 oferentes",
    "6–10 oferentes",
    "Más de 10 oferentes",
    "No informado",
    "Dato no válido"
]


# ------------------------------------------------------------
# ------------------------------------------------------------

tabla_participacion = (
    participacion_oferentes[
        "categoria_participacion"
    ]
    .value_counts()
    .reindex(
        orden_categorias,
        fill_value=0
    )
    .rename(
        "numero_procedimientos"
    )
    .reset_index()
    .rename(
        columns={
            "categoria_participacion": "participacion"
        }
    )
)


# ------------------------------------------------------------
# ------------------------------------------------------------

tabla_participacion[
    "porcentaje"
] = (
    tabla_participacion[
        "numero_procedimientos"
    ]
    / total_procedimientos
    * 100
)


# ------------------------------------------------------------
# ------------------------------------------------------------

total_clasificado = (
    tabla_participacion[
        "numero_procedimientos"
    ]
    .sum()
)


print(
    f"\nProcedimientos clasificados: "
    f"{total_clasificado:,}"
)

print(
    f"Procedimientos esperados: "
    f"{total_procedimientos:,}"
)


if total_clasificado == total_procedimientos:

    print(
        "VALIDACIÓN: Todos los procedimientos fueron "
        "clasificados correctamente."
    )

else:

    print(
        "ADVERTENCIA: La suma de las categorías no coincide "
        "con el total de procedimientos."
    )


# ------------------------------------------------------------
# ------------------------------------------------------------

tabla_participacion_grafico = (
    tabla_participacion[
        tabla_participacion[
            "numero_procedimientos"
        ] > 0
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# ============================================================
# ============================================================

print(
    "\nDISTRIBUCIÓN DE LA PARTICIPACIÓN DE OFERENTES\n"
)

display(
    tabla_participacion_grafico[
        [
            "participacion",
            "numero_procedimientos",
            "porcentaje"
        ]
    ]
)


# ============================================================
# ============================================================

fig, ax = plt.subplots(
    figsize=(11.5, 7.2),
    dpi=150
)


# ------------------------------------------------------------
# ------------------------------------------------------------

barras = ax.barh(
    tabla_participacion_grafico[
        "participacion"
    ],
    tabla_participacion_grafico[
        "numero_procedimientos"
    ],
    color="#4F7D95",
    height=0.65
)


ax.invert_yaxis()


# ------------------------------------------------------------
# ------------------------------------------------------------

maximo = (
    tabla_participacion_grafico[
        "numero_procedimientos"
    ]
    .max()
)


for barra, cantidad, porcentaje in zip(

    barras,

    tabla_participacion_grafico[
        "numero_procedimientos"
    ],

    tabla_participacion_grafico[
        "porcentaje"
    ]
):

    ax.text(
        barra.get_width()
        + maximo * 0.015,

        barra.get_y()
        + barra.get_height() / 2,

        f"{cantidad:,.0f}  ({porcentaje:.1f}%)",

        va="center",
        ha="left",

        fontsize=11,
        fontweight="bold"
    )


# ------------------------------------------------------------
# ------------------------------------------------------------

ax.set_xlabel(
    "Número de procedimientos",
    fontsize=12
)

ax.set_ylabel("")


ax.tick_params(
    axis="y",
    labelsize=11
)

ax.tick_params(
    axis="x",
    labelsize=10
)


# ------------------------------------------------------------
# 15. Líneas guía suaves
# ------------------------------------------------------------

ax.xaxis.grid(
    True,
    linestyle="--",
    alpha=0.20
)

ax.set_axisbelow(True)


# ------------------------------------------------------------
# 16. Eliminar bordes innecesarios
# ------------------------------------------------------------

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)


# ------------------------------------------------------------
# ------------------------------------------------------------

ax.set_xlim(
    0,
    maximo * 1.23
)


# ------------------------------------------------------------
# 18. Ajustar márgenes
# ------------------------------------------------------------

plt.subplots_adjust(
    left=0.23,
    right=0.95,
    top=0.96,
    bottom=0.12
)


# ------------------------------------------------------------
# ------------------------------------------------------------

ruta_figura_1 = os.path.join(
    ruta_figuras,
    "figure_01_bidder_participation_2025.png"
)


fig.savefig(
    ruta_figura_1,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    "\nFigura 1 guardada en:"
)

print(
    ruta_figura_1
)


plt.show()


In [ ]:
# ============================================================
# PASO 6. FIGURA 2
# ============================================================
# Objetivo:
# durante 2025.
#
#
#
#
#
#   montos adjudicados.
#
#
# - Esta figura NO mide concentración, recurrencia ni riesgo.
#
# IMPORTANTE:
# ============================================================


from IPython.display import display


# ------------------------------------------------------------
# ------------------------------------------------------------

distribucion_montos = adjudicaciones_df[
    [
        "ocid",
        "award_amount"
    ]
].copy()


# ------------------------------------------------------------
# ------------------------------------------------------------

distribucion_montos["award_amount"] = pd.to_numeric(
    distribucion_montos["award_amount"],
    errors="coerce"
)


# ------------------------------------------------------------
# ------------------------------------------------------------

total_registros_inicial = len(
    distribucion_montos
)

montos_no_informados = (
    distribucion_montos[
        "award_amount"
    ]
    .isna()
    .sum()
)

montos_cero = (
    distribucion_montos[
        "award_amount"
    ]
    .eq(0)
    .sum()
)

montos_negativos = (
    distribucion_montos[
        "award_amount"
    ]
    .lt(0)
    .sum()
)


print("=" * 65)
print("CONTROL DE MONTOS ADJUDICADOS")
print("=" * 65)

print(
    f"\nRegistros iniciales: "
    f"{total_registros_inicial:,}"
)

print(
    f"Montos no informados/no convertibles: "
    f"{montos_no_informados:,}"
)

print(
    f"Montos iguales a cero: "
    f"{montos_cero:,}"
)

print(
    f"Montos negativos: "
    f"{montos_negativos:,}"
)


# ------------------------------------------------------------
# ------------------------------------------------------------
# positivos.
#
# ------------------------------------------------------------

distribucion_montos = distribucion_montos[
    distribucion_montos[
        "award_amount"
    ].notna()
    &
    (
        distribucion_montos[
            "award_amount"
        ] > 0
    )
].copy()


total_montos_validos = len(
    distribucion_montos
)


print(
    f"\nAdjudicaciones con monto válido > 0: "
    f"{total_montos_validos:,}"
)


# ------------------------------------------------------------
# ------------------------------------------------------------

mediana_monto = (
    distribucion_montos[
        "award_amount"
    ]
    .median()
)

percentil_90 = (
    distribucion_montos[
        "award_amount"
    ]
    .quantile(0.90)
)

percentil_95 = (
    distribucion_montos[
        "award_amount"
    ]
    .quantile(0.95)
)

percentil_99 = (
    distribucion_montos[
        "award_amount"
    ]
    .quantile(0.99)
)

monto_minimo = (
    distribucion_montos[
        "award_amount"
    ]
    .min()
)

monto_maximo = (
    distribucion_montos[
        "award_amount"
    ]
    .max()
)


# ------------------------------------------------------------
# ------------------------------------------------------------
# ------------------------------------------------------------

cantidad_sobre_p99 = (
    distribucion_montos[
        "award_amount"
    ]
    > percentil_99
).sum()


porcentaje_sobre_p99 = (
    cantidad_sobre_p99
    / total_montos_validos
    * 100
)


# ------------------------------------------------------------
# ------------------------------------------------------------
# ------------------------------------------------------------

tabla_resumen_figura_2 = pd.DataFrame(
    {
        "estadistico": [
            "Número de adjudicaciones válidas",
            "Monto mínimo",
            "Mediana",
            "Percentil 90",
            "Percentil 95",
            "Percentil 99",
            "Monto máximo",
            "Adjudicaciones superiores al P99",
            "Porcentaje superior al P99"
        ],

        "valor": [
            total_montos_validos,
            monto_minimo,
            mediana_monto,
            percentil_90,
            percentil_95,
            percentil_99,
            monto_maximo,
            cantidad_sobre_p99,
            porcentaje_sobre_p99
        ]
    }
)


print(
    "\nRESUMEN DESCRIPTIVO DE LOS MONTOS ADJUDICADOS\n"
)

display(
    tabla_resumen_figura_2
)


# ------------------------------------------------------------
# ------------------------------------------------------------

print(
    f"\nMediana: "
    f"USD {mediana_monto:,.2f}"
)

print(
    f"Percentil 99: "
    f"USD {percentil_99:,.2f}"
)

print(
    f"Monto mínimo: "
    f"USD {monto_minimo:,.2f}"
)

print(
    f"Monto máximo: "
    f"USD {monto_maximo:,.2f}"
)

print(
    f"Adjudicaciones superiores al P99: "
    f"{cantidad_sobre_p99:,} "
    f"({porcentaje_sobre_p99:.2f}%)"
)


# ------------------------------------------------------------
# ------------------------------------------------------------
#
# ------------------------------------------------------------

distribucion_montos[
    "award_amount_log10"
] = np.log10(
    distribucion_montos[
        "award_amount"
    ]
)


# ------------------------------------------------------------
# ------------------------------------------------------------

mediana_log10 = np.log10(
    mediana_monto
)

percentil_99_log10 = np.log10(
    percentil_99
)


# ============================================================
# ============================================================

fig, ax = plt.subplots(
    figsize=(11.5, 7.2),
    dpi=150
)


# ------------------------------------------------------------
# ------------------------------------------------------------
# ------------------------------------------------------------

ax.hist(
    distribucion_montos[
        "award_amount_log10"
    ],
    bins=35,
    color="#356F8D",
    edgecolor="white",
    linewidth=0.8,
    alpha=0.92
)


# ------------------------------------------------------------
# ------------------------------------------------------------

ax.axvline(
    mediana_log10,
    color="#D97706",
    linestyle="--",
    linewidth=2.2,
    label=(
        f"Mediana: "
        f"USD {mediana_monto:,.0f}"
    )
)


# ------------------------------------------------------------
# ------------------------------------------------------------

ax.axvline(
    percentil_99_log10,
    color="#7C3AED",
    linestyle=":",
    linewidth=2.3,
    label=(
        f"Percentil 99: "
        f"USD {percentil_99:,.0f}"
    )
)


# ------------------------------------------------------------
# ------------------------------------------------------------
#
# log10(1,000)     = 3
# log10(10,000)    = 4
# log10(100,000)   = 5
# log10(1,000,000) = 6
# ------------------------------------------------------------

exponente_min = int(
    np.floor(
        distribucion_montos[
            "award_amount_log10"
        ].min()
    )
)

exponente_max = int(
    np.ceil(
        distribucion_montos[
            "award_amount_log10"
        ].max()
    )
)


ticks_log = list(
    range(
        exponente_min,
        exponente_max + 1
    )
)


# ------------------------------------------------------------
# ------------------------------------------------------------

def formato_monto_log(exponente):

    valor_monetario = 10 ** exponente

    if valor_monetario >= 1_000_000_000:

        return (
            f"${valor_monetario / 1_000_000_000:g} mil M"
        )

    elif valor_monetario >= 1_000_000:

        return (
            f"${valor_monetario / 1_000_000:g} M"
        )

    elif valor_monetario >= 1_000:

        return (
            f"${valor_monetario / 1_000:g} mil"
        )

    else:

        return (
            f"${valor_monetario:g}"
        )


etiquetas_ticks = [
    formato_monto_log(
        exponente
    )
    for exponente in ticks_log
]


ax.set_xticks(
    ticks_log
)

ax.set_xticklabels(
    etiquetas_ticks,
    fontsize=10.5
)


# ------------------------------------------------------------
# ------------------------------------------------------------

ax.set_xlabel(
    "Monto adjudicado (USD, escala logarítmica base 10)",
    fontsize=12
)

ax.set_ylabel(
    "Número de adjudicaciones",
    fontsize=12
)


# ------------------------------------------------------------
# ------------------------------------------------------------

ax.tick_params(
    axis="y",
    labelsize=10
)


# ------------------------------------------------------------
# 19. Líneas guía suaves
# ------------------------------------------------------------

ax.yaxis.grid(
    True,
    linestyle="--",
    alpha=0.20
)

ax.set_axisbelow(True)


# ------------------------------------------------------------
# 20. Eliminar bordes innecesarios
# ------------------------------------------------------------

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)


# ------------------------------------------------------------
# 21. Leyenda
# ------------------------------------------------------------

ax.legend(
    frameon=False,
    fontsize=10.5,
    loc="upper right"
)


# ------------------------------------------------------------
# 22. Ajustar márgenes
# ------------------------------------------------------------

plt.tight_layout()


# ------------------------------------------------------------
# ------------------------------------------------------------

ruta_figura_2 = os.path.join(
    ruta_figuras,
    "figure_02_award_value_distribution_2025.png"
)


fig.savefig(
    ruta_figura_2,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    "\nFigura 2 guardada en:"
)

print(
    ruta_figura_2
)


plt.show()


In [ ]:
# ============================================================
# PASO 7. FIGURA 3
# ============================================================
# Objetivo:
# Electrónica iniciados durante 2025.
#
#
#
#
# IMPORTANTE:
#   winsorizado al P99.
# - Esta figura NO mide concentración, recurrencia, riesgo
#   ni irregularidad.
# ============================================================


import os
import textwrap
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


# ------------------------------------------------------------
# ------------------------------------------------------------

figura_3_df = adjudicaciones_df[
    [
        "ocid",
        "supplier_id",
        "nombre_proveedor_limpio",
        "award_amount"
    ]
].copy()


# ------------------------------------------------------------
# ------------------------------------------------------------

figura_3_df["award_amount"] = pd.to_numeric(
    figura_3_df["award_amount"],
    errors="coerce"
)


# ------------------------------------------------------------
# 3. Mantener únicamente registros válidos
# ------------------------------------------------------------

figura_3_df = figura_3_df[
    figura_3_df["supplier_id"].notna()
    & figura_3_df["award_amount"].notna()
    & (figura_3_df["award_amount"] > 0)
].copy()


print("=" * 70)
print("CONTROL DE FIGURA 3 - MONTO POR PROVEEDOR")
print("=" * 70)

print(
    f"\nAdjudicaciones válidas utilizadas: "
    f"{len(figura_3_df):,}"
)

print(
    f"Proveedores identificados: "
    f"{figura_3_df['supplier_id'].nunique():,}"
)


# ============================================================
# ============================================================
#
# ============================================================

percentil_99_figura_3 = (
    figura_3_df["award_amount"]
    .quantile(0.99)
)


print(
    f"\nPercentil 99 de adjudicaciones individuales: "
    f"USD {percentil_99_figura_3:,.2f}"
)


# ------------------------------------------------------------
# ------------------------------------------------------------

figura_3_df["supera_p99"] = (
    figura_3_df["award_amount"]
    > percentil_99_figura_3
)


cantidad_supera_p99 = (
    figura_3_df["supera_p99"]
    .sum()
)


porcentaje_supera_p99 = (
    cantidad_supera_p99
    / len(figura_3_df)
    * 100
)


print(
    f"Adjudicaciones superiores al P99: "
    f"{cantidad_supera_p99:,} "
    f"({porcentaje_supera_p99:.2f}%)"
)


# ============================================================
# ============================================================
#
# award_amount permanece intacta.
# ============================================================

figura_3_df[
    "award_amount_winsor_p99"
] = (
    figura_3_df["award_amount"]
    .clip(
        upper=percentil_99_figura_3
    )
)


# ============================================================
# ============================================================
#
# ============================================================

def nombre_representativo(serie):

    serie = serie.dropna()

    if len(serie) == 0:
        return None

    moda = serie.mode()

    if len(moda) > 0:
        return moda.iloc[0]

    return serie.iloc[0]


nombres_proveedor = (
    figura_3_df
    .groupby(
        "supplier_id",
        as_index=False
    )
    .agg(
        nombre_grafico=(
            "nombre_proveedor_limpio",
            nombre_representativo
        )
    )
)


# ============================================================
# ============================================================
#
# ============================================================

proveedores_monto = (
    figura_3_df
    .groupby(
        "supplier_id",
        as_index=False
    )
    .agg(

        monto_total_original=(
            "award_amount",
            "sum"
        ),

        monto_total_winsor_p99=(
            "award_amount_winsor_p99",
            "sum"
        ),

        numero_adjudicaciones=(
            "ocid",
            "nunique"
        ),

        adjudicaciones_superiores_p99=(
            "supera_p99",
            "sum"
        )
    )
)


# ------------------------------------------------------------
# 9. Incorporar nombre limpio
# ------------------------------------------------------------

proveedores_monto = (
    proveedores_monto
    .merge(
        nombres_proveedor,
        on="supplier_id",
        how="left"
    )
)


# Si no existe nombre, utilizar supplier_id

proveedores_monto["nombre_grafico"] = (
    proveedores_monto["nombre_grafico"]
    .fillna(
        proveedores_monto["supplier_id"]
    )
)


# ============================================================
# ============================================================

proveedores_monto[
    "diferencia_winsorizacion"
] = (
    proveedores_monto[
        "monto_total_original"
    ]
    -
    proveedores_monto[
        "monto_total_winsor_p99"
    ]
)


proveedores_monto[
    "reduccion_porcentual"
] = (
    proveedores_monto[
        "diferencia_winsorizacion"
    ]
    /
    proveedores_monto[
        "monto_total_original"
    ]
    * 100
)


# ============================================================
# ============================================================

proveedores_monto[
    "ranking_original"
] = (
    proveedores_monto[
        "monto_total_original"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


proveedores_monto[
    "ranking_winsor_p99"
] = (
    proveedores_monto[
        "monto_total_winsor_p99"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# ============================================================
# 12. SELECCIONAR TOP 10 ROBUSTO
# ============================================================
# acumulado winsorizado al P99.
# ============================================================

top_monto = (
    proveedores_monto
    .sort_values(
        "monto_total_winsor_p99",
        ascending=False
    )
    .head(10)
    .copy()
)


# ============================================================
# ============================================================

top_original_ids = set(
    proveedores_monto
    .sort_values(
        "monto_total_original",
        ascending=False
    )
    .head(10)["supplier_id"]
)


top_winsor_ids = set(
    top_monto["supplier_id"]
)


coincidencias_top10 = len(
    top_original_ids
    .intersection(
        top_winsor_ids
    )
)


print(
    f"\nProveedores que coinciden entre "
    f"Top 10 original y Top 10 winsorizado: "
    f"{coincidencias_top10} de 10"
)


if coincidencias_top10 == 10:

    print(
        "VALIDACIÓN: La winsorización no cambia la "
        "composición del Top 10, aunque puede modificar "
        "el orden y las diferencias monetarias."
    )

else:

    print(
        "RESULTADO: La winsorización modifica la "
        "composición del Top 10, lo que confirma que "
        "los valores extremos influían en el ranking."
    )


# ============================================================
# ============================================================

tabla_figura_3 = (
    top_monto[
        [
            "ranking_winsor_p99",
            "nombre_grafico",
            "monto_total_original",
            "monto_total_winsor_p99",
            "reduccion_porcentual",
            "numero_adjudicaciones",
            "adjudicaciones_superiores_p99",
            "ranking_original"
        ]
    ]
    .sort_values(
        "ranking_winsor_p99"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# ------------------------------------------------------------

tabla_figura_3_mostrar = (
    tabla_figura_3.copy()
)


tabla_figura_3_mostrar = (
    tabla_figura_3_mostrar.rename(
        columns={
            "ranking_winsor_p99":
                "Ranking P99",

            "nombre_grafico":
                "Proveedor",

            "monto_total_original":
                "Monto original",

            "monto_total_winsor_p99":
                "Monto winsorizado P99",

            "reduccion_porcentual":
                "Reducción",

            "numero_adjudicaciones":
                "N.º adjudicaciones",

            "adjudicaciones_superiores_p99":
                "Adjudicaciones > P99",

            "ranking_original":
                "Ranking original"
        }
    )
)



tabla_figura_3_mostrar[
    "Monto original"
] = (
    tabla_figura_3_mostrar[
        "Monto original"
    ]
    .map(
        lambda x: f"USD {x:,.2f}"
    )
)


tabla_figura_3_mostrar[
    "Monto winsorizado P99"
] = (
    tabla_figura_3_mostrar[
        "Monto winsorizado P99"
    ]
    .map(
        lambda x: f"USD {x:,.2f}"
    )
)


tabla_figura_3_mostrar[
    "Reducción"
] = (
    tabla_figura_3_mostrar[
        "Reducción"
    ]
    .map(
        lambda x: f"{x:.2f}%"
    )
)


print(
    "\nTOP 10 PROVEEDORES POR MONTO TOTAL ADJUDICADO"
)

print(
    "Ranking robusto mediante winsorización al P99\n"
)


display(
    tabla_figura_3_mostrar
)


# ============================================================
# ============================================================
# ============================================================

def envolver_texto(
    texto,
    ancho=34
):

    texto = str(
        texto
    )

    return "\n".join(
        textwrap.wrap(
            texto,
            width=ancho,
            break_long_words=False,
            break_on_hyphens=False
        )
    )


top_monto[
    "nombre_grafico_ajustado"
] = (
    top_monto[
        "nombre_grafico"
    ]
    .apply(
        lambda x:
        envolver_texto(
            x,
            ancho=34
        )
    )
)


# ------------------------------------------------------------
# ------------------------------------------------------------

top_monto_grafico = (
    top_monto
    .sort_values(
        "monto_total_winsor_p99",
        ascending=True
    )
    .copy()
)


# ============================================================
# ============================================================

fig, ax = plt.subplots(
    figsize=(13, 8.5),
    dpi=150
)


barras = ax.barh(
    top_monto_grafico[
        "nombre_grafico_ajustado"
    ],
    top_monto_grafico[
        "monto_total_winsor_p99"
    ],
    color="#4C84A3",
    edgecolor="white",
    linewidth=0.8
)


# ============================================================
# ============================================================

max_monto = (
    top_monto_grafico[
        "monto_total_winsor_p99"
    ]
    .max()
)


for barra, valor in zip(

    barras,

    top_monto_grafico[
        "monto_total_winsor_p99"
    ]
):

    ax.text(

        valor
        + max_monto * 0.015,

        barra.get_y()
        + barra.get_height() / 2,

        f"USD {valor / 1_000_000:.2f} M",

        va="center",
        ha="left",

        fontsize=10.5,
        fontweight="bold"
    )


# ============================================================
# ============================================================

ax.set_xlabel(
    "Monto acumulado winsorizado al P99 (USD)",
    fontsize=12
)

ax.set_ylabel("")


# ------------------------------------------------------------
# ------------------------------------------------------------

ax.ticklabel_format(
    style="plain",
    axis="x"
)


ticks = (
    ax.get_xticks()
)


ticks_validos = [
    tick
    for tick in ticks
    if tick >= 0
]


ax.set_xticks(
    ticks_validos
)


ax.set_xticklabels(
    [
        (
            f"{tick / 1_000_000:.0f} M"
            if tick > 0
            else "0"
        )
        for tick in ticks_validos
    ],
    fontsize=10.5
)


# ------------------------------------------------------------
# ------------------------------------------------------------

ax.tick_params(
    axis="y",
    labelsize=10.5
)


# ============================================================
# 19. ESTILO VISUAL
# ============================================================

ax.xaxis.grid(
    True,
    linestyle="--",
    alpha=0.25
)

ax.set_axisbelow(True)


ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

ax.spines[
    "left"
].set_visible(False)


# ------------------------------------------------------------
# ------------------------------------------------------------

ax.set_xlim(
    0,
    max_monto * 1.25
)


# ============================================================
# 20. AJUSTE FINAL
# ============================================================

plt.tight_layout()


# ============================================================
# ============================================================

ruta_figura_3 = os.path.join(
    ruta_figuras,
    "figure_03_top_suppliers_winsorized_value_2025.png"
)


fig.savefig(
    ruta_figura_3,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    "\nFigura 3 guardada en:"
)

print(
    ruta_figura_3
)


plt.show()


In [ ]:
# ============================================================
# PASO 8. FIGURA 4
# ============================================================
# Objetivo:
# Subasta Inversa Electrónica iniciados durante 2025.
#
# CRITERIO METODOLÓGICO:
#
#
#
#   presentación.
#
#
#   no magnitud monetaria.
#
#
# - Esta figura NO mide concentración, recurrencia, riesgo
#   ni irregularidad.
#
#
# ============================================================


import os
import textwrap
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


# ------------------------------------------------------------
# ------------------------------------------------------------

figura_4_df = adjudicaciones_df[
    [
        "ocid",
        "supplier_id",
        "nombre_proveedor_limpio"
    ]
].copy()


# ------------------------------------------------------------
# ------------------------------------------------------------

figura_4_df = figura_4_df[
    figura_4_df["supplier_id"].notna()
    & figura_4_df["ocid"].notna()
].copy()


# ------------------------------------------------------------
# 3. Controles iniciales
# ------------------------------------------------------------

print("=" * 70)
print("CONTROL DE FIGURA 4 - FRECUENCIA DE ADJUDICACIONES")
print("=" * 70)

print(
    f"\nRegistros de adjudicación utilizados: "
    f"{len(figura_4_df):,}"
)

print(
    f"Proveedores identificados: "
    f"{figura_4_df['supplier_id'].nunique():,}"
)

print(
    f"Procedimientos distintos presentes: "
    f"{figura_4_df['ocid'].nunique():,}"
)


# ============================================================
# ============================================================
#
# ============================================================

def nombre_representativo(serie):

    serie = serie.dropna()

    if len(serie) == 0:
        return None

    moda = serie.mode()

    if len(moda) > 0:
        return moda.iloc[0]

    return serie.iloc[0]


nombres_proveedor_figura_4 = (
    figura_4_df
    .groupby(
        "supplier_id",
        as_index=False
    )
    .agg(
        nombre_grafico=(
            "nombre_proveedor_limpio",
            nombre_representativo
        )
    )
)


# ------------------------------------------------------------
# ------------------------------------------------------------

nombres_proveedor_figura_4[
    "nombre_grafico"
] = (
    nombres_proveedor_figura_4[
        "nombre_grafico"
    ]
    .fillna(
        nombres_proveedor_figura_4[
            "supplier_id"
        ]
    )
)


# ============================================================
# ============================================================
# ============================================================

proveedores_frecuencia = (
    figura_4_df
    .groupby(
        "supplier_id",
        as_index=False
    )
    .agg(
        numero_adjudicaciones=(
            "ocid",
            "nunique"
        )
    )
)


# ------------------------------------------------------------
# ------------------------------------------------------------

proveedores_frecuencia = (
    proveedores_frecuencia
    .merge(
        nombres_proveedor_figura_4,
        on="supplier_id",
        how="left"
    )
)


# ============================================================
# ============================================================

proveedores_frecuencia[
    "ranking_frecuencia"
] = (
    proveedores_frecuencia[
        "numero_adjudicaciones"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# ============================================================
# 8. SELECCIONAR TOP 10
# ============================================================

top_frecuencia = (
    proveedores_frecuencia
    .sort_values(
        [
            "numero_adjudicaciones",
            "nombre_grafico"
        ],
        ascending=[
            False,
            True
        ]
    )
    .head(10)
    .copy()
)


# ============================================================
# ============================================================

tabla_figura_4 = (
    top_frecuencia[
        [
            "ranking_frecuencia",
            "nombre_grafico",
            "numero_adjudicaciones"
        ]
    ]
    .sort_values(
        [
            "numero_adjudicaciones",
            "nombre_grafico"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


tabla_figura_4 = (
    tabla_figura_4.rename(
        columns={
            "ranking_frecuencia":
                "Ranking",

            "nombre_grafico":
                "Proveedor",

            "numero_adjudicaciones":
                "Número de adjudicaciones"
        }
    )
)


print(
    "\nTOP 10 PROVEEDORES POR NÚMERO DE ADJUDICACIONES\n"
)



display(
    tabla_figura_4
)


# ============================================================
# ============================================================
# ============================================================

def envolver_texto_figura_4(
    texto,
    ancho=38
):

    texto = str(
        texto
    )

    return "\n".join(
        textwrap.wrap(
            texto,
            width=ancho,
            break_long_words=False,
            break_on_hyphens=False
        )
    )


top_frecuencia[
    "nombre_grafico_ajustado"
] = (
    top_frecuencia[
        "nombre_grafico"
    ]
    .apply(
        lambda x:
        envolver_texto_figura_4(
            x,
            ancho=38
        )
    )
)


# ------------------------------------------------------------
# ------------------------------------------------------------

top_frecuencia_grafico = (
    top_frecuencia
    .sort_values(
        [
            "numero_adjudicaciones",
            "nombre_grafico"
        ],
        ascending=[
            True,
            False
        ]
    )
    .copy()
)


# ============================================================
# ============================================================

fig, ax = plt.subplots(
    figsize=(13, 8.5),
    dpi=150
)


barras = ax.barh(
    top_frecuencia_grafico[
        "nombre_grafico_ajustado"
    ],
    top_frecuencia_grafico[
        "numero_adjudicaciones"
    ],
    color="#4F9A6E",
    edgecolor="white",
    linewidth=0.8
)


# ============================================================
# ============================================================

max_frecuencia = (
    top_frecuencia_grafico[
        "numero_adjudicaciones"
    ]
    .max()
)


for barra, valor in zip(

    barras,

    top_frecuencia_grafico[
        "numero_adjudicaciones"
    ]
):

    ax.text(
        valor
        + max_frecuencia * 0.015,

        barra.get_y()
        + barra.get_height() / 2,

        f"{valor:,.0f}",

        va="center",
        ha="left",

        fontsize=11,
        fontweight="bold"
    )


# ============================================================
# ============================================================

ax.set_xlabel(
    "Número de procedimientos adjudicados",
    fontsize=12
)

ax.set_ylabel("")


ax.tick_params(
    axis="y",
    labelsize=10.5
)

ax.tick_params(
    axis="x",
    labelsize=10.5
)


# ============================================================
# 14. LÍNEAS GUÍA
# ============================================================

ax.xaxis.grid(
    True,
    linestyle="--",
    alpha=0.20
)

ax.set_axisbelow(True)


# ============================================================
# 15. ELIMINAR BORDES INNECESARIOS
# ============================================================

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

ax.spines[
    "left"
].set_visible(False)


# ------------------------------------------------------------
# ------------------------------------------------------------

ax.set_xlim(
    0,
    max_frecuencia * 1.20
)


# ============================================================
# 16. AJUSTAR MÁRGENES
# ============================================================

plt.subplots_adjust(
    left=0.34,
    right=0.95,
    top=0.96,
    bottom=0.12
)


# ============================================================
# ============================================================

ruta_figura_4 = os.path.join(
    ruta_figuras,
    "figure_04_top_suppliers_by_award_count_2025.png"
)


fig.savefig(
    ruta_figura_4,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    "\nFigura 4 guardada en:"
)

print(
    ruta_figura_4
)


plt.show()


In [ ]:
# ============================================================
# PASO 9. FIGURA 5
# ============================================================
# Objetivo:
#
#
#
#
#
# Adicionalmente:
#   correspondientes a esa modalidad.
#
# IMPORTANTE:
# - Recurrencia NO equivale a riesgo.
# - Recurrencia NO demuestra concentración.
# ============================================================


import os
import textwrap
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


# ============================================================
# ============================================================

base_recurrencia = adjudicaciones_df[
    [
        "ocid",
        "buyer_id",
        "supplier_id",
        "nombre_proveedor_limpio",
        "cpc_5",
        "cpc_description",
        "award_date_local"
    ]
].copy()


# ------------------------------------------------------------
# ------------------------------------------------------------

base_recurrencia["fecha_adjudicacion"] = pd.to_datetime(
    base_recurrencia["award_date_local"],
    errors="coerce"
)


# ============================================================
# ============================================================
#
# ============================================================

columnas_tipo_posibles = [
    "tipo_procedimiento",
    "procedure_type",
    "procurement_method_details",
    "tender_procurement_method_details",
    "tender_method_details",
    "modalidad"
]


columna_tipo = next(
    (
        columna
        for columna in columnas_tipo_posibles
        if columna in procedimientos_df.columns
    ),
    None
)


if columna_tipo is not None:

    tipo_procedimiento_df = (
        procedimientos_df[
            [
                "ocid",
                columna_tipo
            ]
        ]
        .drop_duplicates(
            subset=["ocid"]
        )
        .copy()
    )


    base_recurrencia = base_recurrencia.merge(
        tipo_procedimiento_df,
        on="ocid",
        how="left"
    )


    mascara_catalogo = (
        base_recurrencia[
            columna_tipo
        ]
        .astype("string")
        .str.upper()
        .str.contains(
            "CATALOGO|CATÁLOGO",
            na=False
        )
    )


    registros_catalogo = int(
        mascara_catalogo.sum()
    )


    print(
        f"Registros identificados como Catálogo Electrónico: "
        f"{registros_catalogo:,}"
    )


    base_recurrencia = (
        base_recurrencia[
            ~mascara_catalogo
        ]
        .copy()
    )


else:

    print(
        "No se encontró una columna específica de modalidad "
        "en procedimientos_df."
    )

    print(
        "El universo cargado corresponde a Subasta Inversa "
        "Electrónica 2025; no se aplicó un filtro adicional "
        "de Catálogo Electrónico."
    )


# ============================================================
# ============================================================

total_inicial_recurrencia = len(
    base_recurrencia
)


faltantes_recurrencia = (
    base_recurrencia[
        [
            "ocid",
            "buyer_id",
            "supplier_id",
            "cpc_5",
            "fecha_adjudicacion"
        ]
    ]
    .isna()
    .any(axis=1)
    .sum()
)


print("\n" + "=" * 70)
print("CONTROL DEL ANÁLISIS DE RECURRENCIA")
print("=" * 70)

print(
    f"\nRegistros antes de eliminar faltantes: "
    f"{total_inicial_recurrencia:,}"
)

print(
    f"Registros con variables indispensables faltantes: "
    f"{faltantes_recurrencia:,}"
)


# ------------------------------------------------------------
# Mantener únicamente registros completos
# ------------------------------------------------------------

base_recurrencia = (
    base_recurrencia
    .dropna(
        subset=[
            "ocid",
            "buyer_id",
            "supplier_id",
            "cpc_5",
            "fecha_adjudicacion"
        ]
    )
    .copy()
)


# ============================================================
# ============================================================
# ============================================================

base_recurrencia = (
    base_recurrencia
    .drop_duplicates(
        subset=[
            "buyer_id",
            "supplier_id",
            "cpc_5",
            "ocid"
        ]
    )
    .copy()
)


print(
    f"Registros válidos después de depuración: "
    f"{len(base_recurrencia):,}"
)


# ============================================================
# ============================================================

def nombre_representativo_recurrencia(serie):

    serie = serie.dropna()

    if len(serie) == 0:
        return None

    moda = serie.mode()

    if len(moda) > 0:
        return moda.iloc[0]

    return serie.iloc[0]


nombres_compradores = (
    procedimientos_df
    .groupby(
        "buyer_id",
        as_index=False
    )
    .agg(
        buyer_name=(
            "buyer_name",
            nombre_representativo_recurrencia
        )
    )
)


base_recurrencia = (
    base_recurrencia
    .merge(
        nombres_compradores,
        on="buyer_id",
        how="left"
    )
)


# ============================================================
# ============================================================
#
#
#
# - duración observada
# ============================================================

resultados_recurrencia = []


for claves, grupo in base_recurrencia.groupby(
    [
        "buyer_id",
        "supplier_id",
        "cpc_5"
    ]
):

    buyer_id, supplier_id, cpc_5 = claves


    grupo = (
        grupo
        .sort_values(
            "fecha_adjudicacion"
        )
        .reset_index(
            drop=True
        )
    )


    fechas = (
        grupo[
            "fecha_adjudicacion"
        ]
        .tolist()
    )


    inicio = 0

    max_procesos_90d = 0

    fecha_inicio_max = None
    fecha_fin_max = None


    # --------------------------------------------------------
    # Ventana móvil
    # --------------------------------------------------------

    for fin in range(
        len(fechas)
    ):

        while (
            fechas[fin]
            - fechas[inicio]
            > pd.Timedelta(
                days=90
            )
        ):

            inicio += 1


        procesos_ventana = (
            fin
            - inicio
            + 1
        )


        if (
            procesos_ventana
            > max_procesos_90d
        ):

            max_procesos_90d = (
                procesos_ventana
            )

            fecha_inicio_max = (
                fechas[inicio]
            )

            fecha_fin_max = (
                fechas[fin]
            )


    # --------------------------------------------------------
    # --------------------------------------------------------

    nombre_proveedor = (
        nombre_representativo_recurrencia(
            grupo[
                "nombre_proveedor_limpio"
            ]
        )
    )


    if pd.isna(
        nombre_proveedor
    ):

        nombre_proveedor = (
            supplier_id
        )


    # --------------------------------------------------------
    # --------------------------------------------------------

    nombre_comprador = (
        nombre_representativo_recurrencia(
            grupo[
                "buyer_name"
            ]
        )
    )


    if pd.isna(
        nombre_comprador
    ):

        nombre_comprador = (
            buyer_id
        )


    # --------------------------------------------------------
    # Descripción CPC
    # --------------------------------------------------------

    descripcion_cpc = (
        nombre_representativo_recurrencia(
            grupo[
                "cpc_description"
            ]
        )
    )


    if pd.isna(
        descripcion_cpc
    ):

        descripcion_cpc = (
            "Sin descripción"
        )


    # --------------------------------------------------------
    # --------------------------------------------------------

    if (
        fecha_inicio_max is not None
        and fecha_fin_max is not None
    ):

        duracion_ventana_dias = (
            fecha_fin_max
            - fecha_inicio_max
        ).days

    else:

        duracion_ventana_dias = None


    # --------------------------------------------------------
    # --------------------------------------------------------

    resultados_recurrencia.append(
        {
            "buyer_id":
                buyer_id,

            "supplier_id":
                supplier_id,

            "cpc_5":
                cpc_5,

            "buyer_name":
                nombre_comprador,

            "nombre_proveedor":
                nombre_proveedor,

            "cpc_description":
                descripcion_cpc,

            "numero_procesos_total":
                grupo[
                    "ocid"
                ].nunique(),

            "max_procesos_90d":
                max_procesos_90d,

            "fecha_inicio_ventana":
                fecha_inicio_max,

            "fecha_fin_ventana":
                fecha_fin_max,

            "duracion_ventana_dias":
                duracion_ventana_dias
        }
    )


# ------------------------------------------------------------
# Convertir resultados a DataFrame
# ------------------------------------------------------------

recurrencia_df = pd.DataFrame(
    resultados_recurrencia
)


# ============================================================
# ============================================================
# Es decir:
#
# ============================================================

relaciones_recurrentes = (
    recurrencia_df[
        recurrencia_df[
            "max_procesos_90d"
        ] > 3
    ]
    .sort_values(
        [
            "max_procesos_90d",
            "numero_procesos_total",
            "buyer_name",
            "nombre_proveedor"
        ],
        ascending=[
            False,
            False,
            True,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# ============================================================

print(
    f"\nCombinaciones comprador-proveedor-CPC analizadas: "
    f"{len(recurrencia_df):,}"
)


print(
    f"Relaciones recurrentes identificadas: "
    f"{len(relaciones_recurrentes):,}"
)


# ============================================================
# ============================================================

tabla_distribucion_recurrencia = (
    relaciones_recurrentes[
        "max_procesos_90d"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "Máximo de procedimientos en 90 días"
    )
    .reset_index(
        name="Número de relaciones"
    )
)


print(
    "\nDISTRIBUCIÓN DE RELACIONES RECURRENTES\n"
)


display(
    tabla_distribucion_recurrencia
)


# ============================================================
# ============================================================

tabla_relaciones_recurrentes = (
    relaciones_recurrentes[
        [
            "buyer_name",
            "nombre_proveedor",
            "cpc_5",
            "cpc_description",
            "numero_procesos_total",
            "max_procesos_90d",
            "fecha_inicio_ventana",
            "fecha_fin_ventana",
            "duracion_ventana_dias"
        ]
    ]
    .rename(
        columns={
            "buyer_name":
                "Entidad contratante",

            "nombre_proveedor":
                "Proveedor",

            "cpc_5":
                "CPC 5 dígitos",

            "cpc_description":
                "Descripción CPC",

            "numero_procesos_total":
                "Procesos totales",

            "max_procesos_90d":
                "Máximo en 90 días",

            "fecha_inicio_ventana":
                "Inicio ventana",

            "fecha_fin_ventana":
                "Fin ventana",

            "duracion_ventana_dias":
                "Duración observada (días)"
        }
    )
)


print(
    "\nRELACIONES RECURRENTES IDENTIFICADAS\n"
)


display(
    tabla_relaciones_recurrentes
)


# ============================================================
# 11. SELECCIONAR TOP 10
# ============================================================

top_recurrencia = (
    relaciones_recurrentes
    .head(10)
    .copy()
)


# ============================================================
# 12. AJUSTAR NOMBRES COMPLETOS
# ============================================================

def dividir_texto(
    texto,
    ancho=40
):

    if pd.isna(
        texto
    ):

        return (
            "No informado"
        )


    return (
        textwrap.fill(
            str(texto),
            width=ancho,
            break_long_words=False,
            break_on_hyphens=False
        )
    )


# ------------------------------------------------------------
#
# Entidad
# CPC
# ------------------------------------------------------------

top_recurrencia[
    "etiqueta"
] = (
    top_recurrencia
    .apply(
        lambda fila:

        f"{dividir_texto(fila['buyer_name'], 42)}\n"
        f"→ {dividir_texto(fila['nombre_proveedor'], 38)}\n"
        f"CPC {fila['cpc_5']}",

        axis=1
    )
)


# ============================================================
# ============================================================

top_recurrencia_grafico = (
    top_recurrencia
    .sort_values(
        [
            "max_procesos_90d",
            "numero_procesos_total"
        ],
        ascending=[
            True,
            True
        ]
    )
    .copy()
)


# ============================================================
# ============================================================

fig, ax = plt.subplots(
    figsize=(15, 12),
    dpi=150
)


barras = ax.barh(
    top_recurrencia_grafico[
        "etiqueta"
    ],
    top_recurrencia_grafico[
        "max_procesos_90d"
    ],
    color="#527D9B",
    height=0.68
)


# ============================================================
# ============================================================

max_recurrencia = (
    top_recurrencia_grafico[
        "max_procesos_90d"
    ]
    .max()
)


for barra, valor in zip(

    barras,

    top_recurrencia_grafico[
        "max_procesos_90d"
    ]
):

    ax.text(
        valor
        + max_recurrencia * 0.015,

        barra.get_y()
        + barra.get_height() / 2,

        f"{int(valor)}",

        va="center",
        ha="left",

        fontsize=11,
        fontweight="bold"
    )


# ============================================================
# ============================================================

ax.set_xlabel(
    "Máximo de procedimientos dentro de una ventana de 90 días",
    fontsize=11
)

ax.set_ylabel("")


ax.set_xlim(
    0,
    max_recurrencia + 0.8
)


ax.set_xticks(
    range(
        0,
        int(
            max_recurrencia
        ) + 1
    )
)


# ============================================================
# 17. ESTILO VISUAL
# ============================================================

ax.xaxis.grid(
    True,
    linestyle="--",
    alpha=0.20
)

ax.set_axisbelow(True)


ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

ax.spines[
    "left"
].set_visible(False)


ax.tick_params(
    axis="y",
    labelsize=9
)

ax.tick_params(
    axis="x",
    labelsize=10
)


# ------------------------------------------------------------
# ------------------------------------------------------------

plt.subplots_adjust(
    left=0.48,
    right=0.94,
    top=0.97,
    bottom=0.09
)


# ============================================================
# ============================================================

ruta_figura_5 = os.path.join(
    ruta_figuras,
    "figure_05_recurrent_relationships_90_days_2025.png"
)


fig.savefig(
    ruta_figura_5,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    "\nFigura 5 guardada en:"
)

print(
    ruta_figura_5
)


plt.show()


In [ ]:
# ============================================================
# PASO 10. FIGURA 6
# ============================================================
#
# Objetivo:
#
#
# 2. HHI normalizado monetario:
#
# CORRECCIONES METODOLÓGICAS:
#
#   al menos 5 adjudicaciones.
#
#       HHI* = (HHI - 1/n) / (1 - 1/n)
#
#
#   como métricas complementarias.
#
# IMPORTANTE:
#   riesgo o irregularidad.
# ============================================================


import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


# ============================================================
# ============================================================

base_concentracion = adjudicaciones_df[
    [
        "ocid",
        "buyer_id",
        "supplier_id",
        "award_amount"
    ]
].copy()


# ------------------------------------------------------------
# ------------------------------------------------------------

base_concentracion["award_amount"] = pd.to_numeric(
    base_concentracion["award_amount"],
    errors="coerce"
)


print("=" * 75)
print("CONTROL DEL ANÁLISIS DE CONCENTRACIÓN")
print("=" * 75)

print(
    f"\nRegistros iniciales: "
    f"{len(base_concentracion):,}"
)


# ============================================================
# ============================================================
# recurrencia o concentración artificial.
# ============================================================

columnas_modalidad_posibles = [
    "tipo_procedimiento",
    "procedure_type",
    "procurement_method_details",
    "tender_procurement_method_details",
    "tender_method_details",
    "modalidad"
]


columna_modalidad = next(
    (
        columna
        for columna in columnas_modalidad_posibles
        if columna in procedimientos_df.columns
    ),
    None
)


if columna_modalidad is not None:

    modalidad_df = (
        procedimientos_df[
            [
                "ocid",
                columna_modalidad
            ]
        ]
        .drop_duplicates(
            subset=["ocid"]
        )
        .copy()
    )


    base_concentracion = (
        base_concentracion
        .merge(
            modalidad_df,
            on="ocid",
            how="left"
        )
    )


    mascara_catalogo = (
        base_concentracion[
            columna_modalidad
        ]
        .astype("string")
        .str.upper()
        .str.contains(
            r"CAT[ÁA]LOGO",
            regex=True,
            na=False
        )
    )


    registros_catalogo = int(
        mascara_catalogo.sum()
    )


    print(
        f"\nRegistros identificados como Catálogo Electrónico: "
        f"{registros_catalogo:,}"
    )


    base_concentracion = (
        base_concentracion[
            ~mascara_catalogo
        ]
        .copy()
    )


else:

    print(
        "\nNo se encontró una columna específica de modalidad "
        "en procedimientos_df."
    )

    print(
        "El universo cargado corresponde a Subasta Inversa "
        "Electrónica 2025; no se aplicó un filtro adicional "
        "de Catálogo Electrónico."
    )


# ============================================================
# ============================================================
# ============================================================

base_concentracion = base_concentracion[
    base_concentracion["ocid"].notna()
    & base_concentracion["buyer_id"].notna()
    & base_concentracion["supplier_id"].notna()
    & base_concentracion["award_amount"].notna()
    & (base_concentracion["award_amount"] >= 5000)
].copy()


print(
    f"\nRegistros con monto >= USD 5.000: "
    f"{len(base_concentracion):,}"
)


# ============================================================
# 4. ELIMINAR DUPLICADOS EXACTOS
# ============================================================
#
# diferente.
# ============================================================

base_concentracion = (
    base_concentracion
    .drop_duplicates(
        subset=[
            "ocid",
            "buyer_id",
            "supplier_id",
            "award_amount"
        ]
    )
    .copy()
)


# ============================================================
# ============================================================
# Esto permite:
#
# ============================================================

procedimiento_proveedor = (
    base_concentracion
    .groupby(
        [
            "ocid",
            "buyer_id",
            "supplier_id"
        ],
        as_index=False
    )
    .agg(
        monto_procedimiento=(
            "award_amount",
            "sum"
        )
    )
)


# ============================================================
# ============================================================

resumen_entidades = (
    procedimiento_proveedor
    .groupby(
        "buyer_id",
        as_index=False
    )
    .agg(
        numero_adjudicaciones=(
            "ocid",
            "nunique"
        ),

        numero_proveedores=(
            "supplier_id",
            "nunique"
        )
    )
)


entidades_elegibles = (
    resumen_entidades[
        resumen_entidades[
            "numero_adjudicaciones"
        ] >= 5
    ]
    .copy()
)


print(
    f"\nEntidades con al menos 5 adjudicaciones: "
    f"{len(entidades_elegibles):,}"
)


# ------------------------------------------------------------
# Mantener únicamente compradores elegibles
# ------------------------------------------------------------

procedimiento_proveedor = (
    procedimiento_proveedor[
        procedimiento_proveedor[
            "buyer_id"
        ].isin(
            entidades_elegibles[
                "buyer_id"
            ]
        )
    ]
    .copy()
)


# ============================================================
# ============================================================

relacion_concentracion = (
    procedimiento_proveedor
    .groupby(
        [
            "buyer_id",
            "supplier_id"
        ],
        as_index=False
    )
    .agg(
        frecuencia=(
            "ocid",
            "nunique"
        ),

        monto_total=(
            "monto_procedimiento",
            "sum"
        )
    )
)


# ============================================================
# ============================================================

totales_entidad = (
    relacion_concentracion
    .groupby(
        "buyer_id",
        as_index=False
    )
    .agg(
        total_adjudicaciones=(
            "frecuencia",
            "sum"
        ),

        total_monto=(
            "monto_total",
            "sum"
        ),

        numero_proveedores=(
            "supplier_id",
            "nunique"
        )
    )
)


relacion_concentracion = (
    relacion_concentracion
    .merge(
        totales_entidad,
        on="buyer_id",
        how="left"
    )
)


# ============================================================
# ============================================================

relacion_concentracion[
    "participacion_frecuencia"
] = (
    relacion_concentracion[
        "frecuencia"
    ]
    /
    relacion_concentracion[
        "total_adjudicaciones"
    ]
)


relacion_concentracion[
    "participacion_monetaria"
] = (
    relacion_concentracion[
        "monto_total"
    ]
    /
    relacion_concentracion[
        "total_monto"
    ]
)


# ============================================================
# ============================================================

relacion_concentracion[
    "hhi_comp_frecuencia"
] = (
    relacion_concentracion[
        "participacion_frecuencia"
    ] ** 2
)


relacion_concentracion[
    "hhi_comp_monetario"
] = (
    relacion_concentracion[
        "participacion_monetaria"
    ] ** 2
)


# ============================================================
# ============================================================

hhi_entidad = (
    relacion_concentracion
    .groupby(
        "buyer_id",
        as_index=False
    )
    .agg(
        hhi_frecuencia=(
            "hhi_comp_frecuencia",
            "sum"
        ),

        hhi_monetario=(
            "hhi_comp_monetario",
            "sum"
        ),

        numero_proveedores=(
            "supplier_id",
            "nunique"
        ),

        numero_adjudicaciones=(
            "frecuencia",
            "sum"
        ),

        monto_total_adjudicado=(
            "monto_total",
            "sum"
        )
    )
)


# ============================================================
# 12. NORMALIZAR HHI
# ============================================================
#
# HHI* = (HHI - 1/n) / (1 - 1/n)
#
#
# Si n = 1:
# HHI* NO está definido.
# ============================================================

def normalizar_hhi(
    hhi,
    n
):

    if n <= 1:
        return np.nan


    hhi_normalizado = (
        (
            hhi
            - (1 / n)
        )
        /
        (
            1
            - (1 / n)
        )
    )


    return float(
        np.clip(
            hhi_normalizado,
            0,
            1
        )
    )


hhi_entidad[
    "hhi_frecuencia_normalizado"
] = (
    hhi_entidad.apply(
        lambda fila:
        normalizar_hhi(
            fila[
                "hhi_frecuencia"
            ],
            fila[
                "numero_proveedores"
            ]
        ),
        axis=1
    )
)


hhi_entidad[
    "hhi_monetario_normalizado"
] = (
    hhi_entidad.apply(
        lambda fila:
        normalizar_hhi(
            fila[
                "hhi_monetario"
            ],
            fila[
                "numero_proveedores"
            ]
        ),
        axis=1
    )
)


# ============================================================
# ============================================================
# CR4:
#
#
# - frecuencia
# ============================================================

def calcular_cr4(grupo):

    cr4_frecuencia = (
        grupo[
            "participacion_frecuencia"
        ]
        .nlargest(4)
        .sum()
    )


    cr4_monetario = (
        grupo[
            "participacion_monetaria"
        ]
        .nlargest(4)
        .sum()
    )


    return pd.Series(
        {
            "cr4_frecuencia":
                cr4_frecuencia,

            "cr4_monetario":
                cr4_monetario
        }
    )


cr4_entidad = (
    relacion_concentracion
    .groupby(
        "buyer_id"
    )
    .apply(
        calcular_cr4,
        include_groups=False
    )
    .reset_index()
)


# ============================================================
# ============================================================
# Entropía normalizada:
#
# H = -SUM(p * ln(p)) / ln(n)
#
# Valores cercanos a 1:
#
# Valores cercanos a 0:
# menor diversidad.
#
# ============================================================

def entropia_normalizada(
    participaciones
):

    participaciones = (
        np.asarray(
            participaciones,
            dtype=float
        )
    )


    participaciones = (
        participaciones[
            participaciones > 0
        ]
    )


    n = len(
        participaciones
    )


    if n <= 1:
        return np.nan


    entropia = (
        -np.sum(
            participaciones
            * np.log(
                participaciones
            )
        )
    )


    return float(
        entropia
        / np.log(n)
    )


entropia_entidad = (
    relacion_concentracion
    .groupby(
        "buyer_id"
    )
    .apply(
        lambda grupo:
        pd.Series(
            {
                "entropia_frecuencia":
                    entropia_normalizada(
                        grupo[
                            "participacion_frecuencia"
                        ]
                    ),

                "entropia_monetaria":
                    entropia_normalizada(
                        grupo[
                            "participacion_monetaria"
                        ]
                    )
            }
        ),
        include_groups=False
    )
    .reset_index()
)


# ============================================================
# ============================================================

hhi_entidad = (
    hhi_entidad
    .merge(
        cr4_entidad,
        on="buyer_id",
        how="left"
    )
    .merge(
        entropia_entidad,
        on="buyer_id",
        how="left"
    )
)


# ============================================================
# ============================================================

def nombre_representativo_concentracion(
    serie
):

    serie = (
        serie
        .dropna()
    )


    if len(serie) == 0:
        return None


    moda = (
        serie.mode()
    )


    if len(moda) > 0:
        return moda.iloc[0]


    return serie.iloc[0]


nombres_entidades = (
    procedimientos_df
    .groupby(
        "buyer_id",
        as_index=False
    )
    .agg(
        buyer_name=(
            "buyer_name",
            nombre_representativo_concentracion
        )
    )
)


hhi_entidad = (
    hhi_entidad
    .merge(
        nombres_entidades,
        on="buyer_id",
        how="left"
    )
)


# ============================================================
# ============================================================

casos_un_proveedor = (
    hhi_entidad[
        hhi_entidad[
            "numero_proveedores"
        ] == 1
    ]
    .copy()
)


print(
    f"\nEntidades elegibles con un solo proveedor "
    f"(HHI* no definido): "
    f"{len(casos_un_proveedor):,}"
)


# ============================================================
# ============================================================

hhi_figura = (
    hhi_entidad
    .dropna(
        subset=[
            "hhi_frecuencia_normalizado",
            "hhi_monetario_normalizado"
        ]
    )
    .copy()
)


print(
    f"Entidades representadas en la Figura 6: "
    f"{len(hhi_figura):,}"
)


# ============================================================
# ============================================================

mediana_hhi_frecuencia = (
    hhi_figura[
        "hhi_frecuencia_normalizado"
    ]
    .median()
)


mediana_hhi_monetario = (
    hhi_figura[
        "hhi_monetario_normalizado"
    ]
    .median()
)


print(
    f"\nMediana HHI* por frecuencia: "
    f"{mediana_hhi_frecuencia:.3f}"
)

print(
    f"Mediana HHI* monetario: "
    f"{mediana_hhi_monetario:.3f}"
)


# ============================================================
# 20. DIFERENCIA ENTRE AMBOS HHI
# ============================================================
# Diferencia positiva:
#
# HHI monetario > HHI frecuencia.
#
# Diferencia negativa:
#
# HHI frecuencia > HHI monetario.
#
# ============================================================

hhi_figura[
    "diferencia_hhi"
] = (
    hhi_figura[
        "hhi_monetario_normalizado"
    ]
    -
    hhi_figura[
        "hhi_frecuencia_normalizado"
    ]
)


hhi_figura[
    "diferencia_absoluta"
] = (
    hhi_figura[
        "diferencia_hhi"
    ]
    .abs()
)


# ============================================================
# ============================================================

top_diferencias_tabla = (
    hhi_figura
    .sort_values(
        "diferencia_absoluta",
        ascending=False
    )
    .head(10)
    .copy()
)


tabla_figura_6 = (
    top_diferencias_tabla[
        [
            "buyer_name",
            "numero_adjudicaciones",
            "numero_proveedores",
            "hhi_frecuencia_normalizado",
            "hhi_monetario_normalizado",
            "diferencia_hhi",
            "cr4_frecuencia",
            "cr4_monetario",
            "entropia_frecuencia",
            "entropia_monetaria"
        ]
    ]
    .rename(
        columns={
            "buyer_name":
                "Entidad contratante",

            "numero_adjudicaciones":
                "Adjudicaciones",

            "numero_proveedores":
                "Proveedores",

            "hhi_frecuencia_normalizado":
                "HHI* frecuencia",

            "hhi_monetario_normalizado":
                "HHI* monetario",

            "diferencia_hhi":
                "Diferencia HHI*",

            "cr4_frecuencia":
                "CR4 frecuencia",

            "cr4_monetario":
                "CR4 monetario",

            "entropia_frecuencia":
                "Entropía frecuencia",

            "entropia_monetaria":
                "Entropía monetaria"
        }
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# ------------------------------------------------------------

columnas_decimales = [
    "HHI* frecuencia",
    "HHI* monetario",
    "Diferencia HHI*",
    "CR4 frecuencia",
    "CR4 monetario",
    "Entropía frecuencia",
    "Entropía monetaria"
]


tabla_figura_6[
    columnas_decimales
] = (
    tabla_figura_6[
        columnas_decimales
    ]
    .round(3)
)


print(
    "\nENTIDADES CON MAYOR DIFERENCIA ENTRE "
    "CONCENTRACIÓN MONETARIA Y POR FRECUENCIA\n"
)


display(
    tabla_figura_6
)


# ============================================================
# ============================================================
#
# ============================================================

fig, ax = plt.subplots(
    figsize=(10.5, 9.5),
    dpi=150
)


ax.scatter(
    hhi_figura[
        "hhi_frecuencia_normalizado"
    ],
    hhi_figura[
        "hhi_monetario_normalizado"
    ],
    s=55,
    alpha=0.65,
    color="#3D7EA6",
    edgecolor="white",
    linewidth=0.5
)


# ============================================================
# ============================================================
#
# Encima:
# concentración monetaria > frecuencia.
#
# Debajo:
# ============================================================

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.4,
    color="#666666"
)


# ============================================================
# ============================================================

top_5_diferencias = (
    hhi_figura
    .sort_values(
        "diferencia_absoluta",
        ascending=False
    )
    .head(5)
    .copy()
)


def dividir_nombre_figura_6(
    texto,
    ancho=28
):

    if pd.isna(texto):
        return "No informado"


    return "\n".join(
        textwrap.wrap(
            str(texto),
            width=ancho,
            break_long_words=False,
            break_on_hyphens=False
        )
    )


desplazamientos = [
    (16, 22),
    (16, -12),
    (16, 18),
    (16, -25),
    (16, 8)
]


for (
    (_, fila),
    desplazamiento
) in zip(
    top_5_diferencias.iterrows(),
    desplazamientos
):

    nombre = (
        dividir_nombre_figura_6(
            fila[
                "buyer_name"
            ]
        )
    )


    ax.annotate(
        nombre,

        (
            fila[
                "hhi_frecuencia_normalizado"
            ],

            fila[
                "hhi_monetario_normalizado"
            ]
        ),

        xytext=desplazamiento,

        textcoords="offset points",

        fontsize=8,

        ha="left",
        va="center",

        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="none",
            alpha=0.82
        ),

        arrowprops=dict(
            arrowstyle="-",
            linewidth=0.7,
            color="#777777"
        )
    )


# ============================================================
# ============================================================

ax.set_xlabel(
    "HHI normalizado por frecuencia",
    fontsize=11
)


ax.set_ylabel(
    "HHI normalizado por monto adjudicado",
    fontsize=11
)


ax.set_xlim(
    -0.02,
    1.02
)

ax.set_ylim(
    -0.02,
    1.02
)


ax.set_aspect(
    "equal",
    adjustable="box"
)


# ============================================================
# 26. ESTILO VISUAL
# ============================================================

ax.grid(
    True,
    linestyle="--",
    alpha=0.20
)

ax.set_axisbelow(True)


ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)


ax.tick_params(
    axis="both",
    labelsize=9.5
)


plt.tight_layout()


# ============================================================
# ============================================================

ruta_figura_6 = os.path.join(
    ruta_figuras,
    "figure_06_normalized_hhi_value_frequency_2025.png"
)


fig.savefig(
    ruta_figura_6,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    "\nFigura 6 guardada en:"
)

print(
    ruta_figura_6
)


plt.show()


In [ ]:
# ============================================================
# PASO 10. FIGURA 6
# ============================================================
#
# Objetivo:
#
#
# 2. HHI normalizado monetario:
#
# CORRECCIONES METODOLÓGICAS:
#
#   al menos 5 adjudicaciones elegibles.
#
#       HHI* = (HHI - 1/n) / (1 - 1/n)
#
#
#   como métricas complementarias.
#
# IMPORTANTE:
#   riesgo o irregularidad.
# ============================================================


import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


# ============================================================
# ============================================================

base_concentracion = adjudicaciones_df[
    [
        "ocid",
        "buyer_id",
        "supplier_id",
        "award_amount"
    ]
].copy()


# ------------------------------------------------------------
# ------------------------------------------------------------

base_concentracion["award_amount"] = pd.to_numeric(
    base_concentracion["award_amount"],
    errors="coerce"
)


print("=" * 75)
print("CONTROL DEL ANÁLISIS DE CONCENTRACIÓN")
print("=" * 75)

print(
    f"\nRegistros iniciales: "
    f"{len(base_concentracion):,}"
)


# ============================================================
# ============================================================
# recurrencia o concentración artificial.
# ============================================================

columnas_modalidad_posibles = [
    "tipo_procedimiento",
    "procedure_type",
    "procurement_method_details",
    "tender_procurement_method_details",
    "tender_method_details",
    "modalidad"
]


columna_modalidad = next(
    (
        columna
        for columna in columnas_modalidad_posibles
        if columna in procedimientos_df.columns
    ),
    None
)


if columna_modalidad is not None:

    modalidad_df = (
        procedimientos_df[
            [
                "ocid",
                columna_modalidad
            ]
        ]
        .drop_duplicates(
            subset=["ocid"]
        )
        .copy()
    )


    base_concentracion = (
        base_concentracion
        .merge(
            modalidad_df,
            on="ocid",
            how="left"
        )
    )


    mascara_catalogo = (
        base_concentracion[
            columna_modalidad
        ]
        .astype("string")
        .str.upper()
        .str.contains(
            r"CAT[ÁA]LOGO",
            regex=True,
            na=False
        )
    )


    registros_catalogo = int(
        mascara_catalogo.sum()
    )


    print(
        f"\nRegistros identificados como Catálogo Electrónico: "
        f"{registros_catalogo:,}"
    )


    base_concentracion = (
        base_concentracion[
            ~mascara_catalogo
        ]
        .copy()
    )


else:

    print(
        "\nNo se encontró una columna específica de modalidad "
        "en procedimientos_df."
    )

    print(
        "El universo cargado corresponde a Subasta Inversa "
        "Electrónica 2025; no se aplicó un filtro adicional "
        "de Catálogo Electrónico."
    )


# ============================================================
# ============================================================
# ============================================================

base_concentracion = base_concentracion[
    base_concentracion["ocid"].notna()
    & base_concentracion["buyer_id"].notna()
    & base_concentracion["supplier_id"].notna()
    & base_concentracion["award_amount"].notna()
    & (base_concentracion["award_amount"] >= 5000)
].copy()


print(
    f"\nRegistros con monto >= USD 5.000: "
    f"{len(base_concentracion):,}"
)


# ============================================================
# 4. ELIMINAR DUPLICADOS EXACTOS
# ============================================================
#
# diferente.
# ============================================================

base_concentracion = (
    base_concentracion
    .drop_duplicates(
        subset=[
            "ocid",
            "buyer_id",
            "supplier_id",
            "award_amount"
        ]
    )
    .copy()
)


# ============================================================
# ============================================================
# Esto permite:
#
# ============================================================

procedimiento_proveedor = (
    base_concentracion
    .groupby(
        [
            "ocid",
            "buyer_id",
            "supplier_id"
        ],
        as_index=False
    )
    .agg(
        monto_procedimiento=(
            "award_amount",
            "sum"
        )
    )
)


print(
    f"Registros procedimiento-proveedor después del filtro: "
    f"{len(procedimiento_proveedor):,}"
)


# ============================================================
# ============================================================
# IMPORTANTE:
# ============================================================

resumen_entidades = (
    procedimiento_proveedor
    .groupby(
        "buyer_id",
        as_index=False
    )
    .agg(
        numero_adjudicaciones=(
            "ocid",
            "nunique"
        ),

        numero_proveedores=(
            "supplier_id",
            "nunique"
        )
    )
)


entidades_elegibles = (
    resumen_entidades[
        resumen_entidades[
            "numero_adjudicaciones"
        ] >= 5
    ]
    .copy()
)


print(
    f"\nEntidades con al menos 5 adjudicaciones "
    f"de USD 5.000 o más: "
    f"{len(entidades_elegibles):,}"
)


# ============================================================
# ============================================================
# Este bloque confirma explícitamente:
#
#    haya sido incorporada al análisis.
# ============================================================

print("\n" + "=" * 75)
print("VALIDACIÓN DE CRITERIOS DE CONCENTRACIÓN")
print("=" * 75)

print(
    f"Registros procedimiento-proveedor con monto >= USD 5.000: "
    f"{len(procedimiento_proveedor):,}"
)

print(
    f"Compradores después del filtro de USD 5.000: "
    f"{resumen_entidades['buyer_id'].nunique():,}"
)

print(
    f"Compradores elegibles con >= 5 adjudicaciones: "
    f"{len(entidades_elegibles):,}"
)

print(
    f"Mínimo de adjudicaciones entre compradores elegibles: "
    f"{entidades_elegibles['numero_adjudicaciones'].min():,}"
)

print(
    f"Máximo de adjudicaciones entre compradores elegibles: "
    f"{entidades_elegibles['numero_adjudicaciones'].max():,}"
)

compradores_menos_5 = (
    entidades_elegibles["numero_adjudicaciones"] < 5
).sum()

print(
    "Compradores elegibles con menos de 5 adjudicaciones:",
    compradores_menos_5
)


# ------------------------------------------------------------
# Mantener únicamente compradores elegibles
# ------------------------------------------------------------

procedimiento_proveedor = (
    procedimiento_proveedor[
        procedimiento_proveedor[
            "buyer_id"
        ].isin(
            entidades_elegibles[
                "buyer_id"
            ]
        )
    ]
    .copy()
)


print(
    f"Registros procedimiento-proveedor correspondientes "
    f"a compradores elegibles: "
    f"{len(procedimiento_proveedor):,}"
)


# ============================================================
# ============================================================

relacion_concentracion = (
    procedimiento_proveedor
    .groupby(
        [
            "buyer_id",
            "supplier_id"
        ],
        as_index=False
    )
    .agg(
        frecuencia=(
            "ocid",
            "nunique"
        ),

        monto_total=(
            "monto_procedimiento",
            "sum"
        )
    )
)


# ============================================================
# ============================================================

totales_entidad = (
    relacion_concentracion
    .groupby(
        "buyer_id",
        as_index=False
    )
    .agg(
        total_adjudicaciones=(
            "frecuencia",
            "sum"
        ),

        total_monto=(
            "monto_total",
            "sum"
        ),

        numero_proveedores=(
            "supplier_id",
            "nunique"
        )
    )
)


relacion_concentracion = (
    relacion_concentracion
    .merge(
        totales_entidad,
        on="buyer_id",
        how="left"
    )
)


# ============================================================
# ============================================================

relacion_concentracion[
    "participacion_frecuencia"
] = (
    relacion_concentracion[
        "frecuencia"
    ]
    /
    relacion_concentracion[
        "total_adjudicaciones"
    ]
)


relacion_concentracion[
    "participacion_monetaria"
] = (
    relacion_concentracion[
        "monto_total"
    ]
    /
    relacion_concentracion[
        "total_monto"
    ]
)


# ============================================================
# ============================================================

relacion_concentracion[
    "hhi_comp_frecuencia"
] = (
    relacion_concentracion[
        "participacion_frecuencia"
    ] ** 2
)


relacion_concentracion[
    "hhi_comp_monetario"
] = (
    relacion_concentracion[
        "participacion_monetaria"
    ] ** 2
)


# ============================================================
# ============================================================

hhi_entidad = (
    relacion_concentracion
    .groupby(
        "buyer_id",
        as_index=False
    )
    .agg(
        hhi_frecuencia=(
            "hhi_comp_frecuencia",
            "sum"
        ),

        hhi_monetario=(
            "hhi_comp_monetario",
            "sum"
        ),

        numero_proveedores=(
            "supplier_id",
            "nunique"
        ),

        numero_adjudicaciones=(
            "frecuencia",
            "sum"
        ),

        monto_total_adjudicado=(
            "monto_total",
            "sum"
        )
    )
)


# ============================================================
# 12. NORMALIZAR HHI
# ============================================================
#
# HHI* = (HHI - 1/n) / (1 - 1/n)
#
#
# Si n = 1:
# HHI* NO está definido.
# ============================================================

def normalizar_hhi(
    hhi,
    n
):

    if n <= 1:
        return np.nan


    hhi_normalizado = (
        (
            hhi
            - (1 / n)
        )
        /
        (
            1
            - (1 / n)
        )
    )


    return float(
        np.clip(
            hhi_normalizado,
            0,
            1
        )
    )


hhi_entidad[
    "hhi_frecuencia_normalizado"
] = (
    hhi_entidad.apply(
        lambda fila:
        normalizar_hhi(
            fila[
                "hhi_frecuencia"
            ],
            fila[
                "numero_proveedores"
            ]
        ),
        axis=1
    )
)


hhi_entidad[
    "hhi_monetario_normalizado"
] = (
    hhi_entidad.apply(
        lambda fila:
        normalizar_hhi(
            fila[
                "hhi_monetario"
            ],
            fila[
                "numero_proveedores"
            ]
        ),
        axis=1
    )
)


# ============================================================
# ============================================================
# CR4:
#
#
# - frecuencia
# ============================================================

def calcular_cr4(grupo):

    cr4_frecuencia = (
        grupo[
            "participacion_frecuencia"
        ]
        .nlargest(4)
        .sum()
    )


    cr4_monetario = (
        grupo[
            "participacion_monetaria"
        ]
        .nlargest(4)
        .sum()
    )


    return pd.Series(
        {
            "cr4_frecuencia":
                cr4_frecuencia,

            "cr4_monetario":
                cr4_monetario
        }
    )


cr4_entidad = (
    relacion_concentracion
    .groupby(
        "buyer_id"
    )
    .apply(
        calcular_cr4,
        include_groups=False
    )
    .reset_index()
)


# ============================================================
# ============================================================
# Entropía normalizada:
#
# H = -SUM(p * ln(p)) / ln(n)
#
# Valores cercanos a 1:
#
# Valores cercanos a 0:
# menor diversidad.
#
# ============================================================

def entropia_normalizada(
    participaciones
):

    participaciones = (
        np.asarray(
            participaciones,
            dtype=float
        )
    )


    participaciones = (
        participaciones[
            participaciones > 0
        ]
    )


    n = len(
        participaciones
    )


    if n <= 1:
        return np.nan


    entropia = (
        -np.sum(
            participaciones
            * np.log(
                participaciones
            )
        )
    )


    return float(
        entropia
        / np.log(n)
    )


entropia_entidad = (
    relacion_concentracion
    .groupby(
        "buyer_id"
    )
    .apply(
        lambda grupo:
        pd.Series(
            {
                "entropia_frecuencia":
                    entropia_normalizada(
                        grupo[
                            "participacion_frecuencia"
                        ]
                    ),

                "entropia_monetaria":
                    entropia_normalizada(
                        grupo[
                            "participacion_monetaria"
                        ]
                    )
            }
        ),
        include_groups=False
    )
    .reset_index()
)


# ============================================================
# ============================================================

hhi_entidad = (
    hhi_entidad
    .merge(
        cr4_entidad,
        on="buyer_id",
        how="left"
    )
    .merge(
        entropia_entidad,
        on="buyer_id",
        how="left"
    )
)


# ============================================================
# ============================================================

def nombre_representativo_concentracion(
    serie
):

    serie = (
        serie
        .dropna()
    )


    if len(serie) == 0:
        return None


    moda = (
        serie.mode()
    )


    if len(moda) > 0:
        return moda.iloc[0]


    return serie.iloc[0]


nombres_entidades = (
    procedimientos_df
    .groupby(
        "buyer_id",
        as_index=False
    )
    .agg(
        buyer_name=(
            "buyer_name",
            nombre_representativo_concentracion
        )
    )
)


hhi_entidad = (
    hhi_entidad
    .merge(
        nombres_entidades,
        on="buyer_id",
        how="left"
    )
)


# ============================================================
# ============================================================

casos_un_proveedor = (
    hhi_entidad[
        hhi_entidad[
            "numero_proveedores"
        ] == 1
    ]
    .copy()
)


print(
    f"\nEntidades elegibles con un solo proveedor "
    f"(HHI* no definido): "
    f"{len(casos_un_proveedor):,}"
)


# ============================================================
# ============================================================

hhi_figura = (
    hhi_entidad
    .dropna(
        subset=[
            "hhi_frecuencia_normalizado",
            "hhi_monetario_normalizado"
        ]
    )
    .copy()
)


print(
    f"Entidades representadas en la Figura 6: "
    f"{len(hhi_figura):,}"
)


# ============================================================
# ============================================================

mediana_hhi_frecuencia = (
    hhi_figura[
        "hhi_frecuencia_normalizado"
    ]
    .median()
)


mediana_hhi_monetario = (
    hhi_figura[
        "hhi_monetario_normalizado"
    ]
    .median()
)


print(
    f"\nMediana HHI* por frecuencia: "
    f"{mediana_hhi_frecuencia:.3f}"
)

print(
    f"Mediana HHI* monetario: "
    f"{mediana_hhi_monetario:.3f}"
)


# ============================================================
# 20. DIFERENCIA ENTRE AMBOS HHI
# ============================================================
# Diferencia positiva:
#
# HHI monetario > HHI frecuencia.
#
# Diferencia negativa:
#
# HHI frecuencia > HHI monetario.
#
# ============================================================

hhi_figura[
    "diferencia_hhi"
] = (
    hhi_figura[
        "hhi_monetario_normalizado"
    ]
    -
    hhi_figura[
        "hhi_frecuencia_normalizado"
    ]
)


hhi_figura[
    "diferencia_absoluta"
] = (
    hhi_figura[
        "diferencia_hhi"
    ]
    .abs()
)


# ============================================================
# ============================================================

top_diferencias_tabla = (
    hhi_figura
    .sort_values(
        "diferencia_absoluta",
        ascending=False
    )
    .head(10)
    .copy()
)


tabla_figura_6 = (
    top_diferencias_tabla[
        [
            "buyer_name",
            "numero_adjudicaciones",
            "numero_proveedores",
            "hhi_frecuencia_normalizado",
            "hhi_monetario_normalizado",
            "diferencia_hhi",
            "cr4_frecuencia",
            "cr4_monetario",
            "entropia_frecuencia",
            "entropia_monetaria"
        ]
    ]
    .rename(
        columns={
            "buyer_name":
                "Entidad contratante",

            "numero_adjudicaciones":
                "Adjudicaciones",

            "numero_proveedores":
                "Proveedores",

            "hhi_frecuencia_normalizado":
                "HHI* frecuencia",

            "hhi_monetario_normalizado":
                "HHI* monetario",

            "diferencia_hhi":
                "Diferencia HHI*",

            "cr4_frecuencia":
                "CR4 frecuencia",

            "cr4_monetario":
                "CR4 monetario",

            "entropia_frecuencia":
                "Entropía frecuencia",

            "entropia_monetaria":
                "Entropía monetaria"
        }
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# ------------------------------------------------------------

columnas_decimales = [
    "HHI* frecuencia",
    "HHI* monetario",
    "Diferencia HHI*",
    "CR4 frecuencia",
    "CR4 monetario",
    "Entropía frecuencia",
    "Entropía monetaria"
]


tabla_figura_6[
    columnas_decimales
] = (
    tabla_figura_6[
        columnas_decimales
    ]
    .round(3)
)


print(
    "\nENTIDADES CON MAYOR DIFERENCIA ENTRE "
    "CONCENTRACIÓN MONETARIA Y POR FRECUENCIA\n"
)


display(
    tabla_figura_6
)


# ============================================================
# ============================================================
#
# ============================================================

fig, ax = plt.subplots(
    figsize=(10.5, 9.5),
    dpi=150
)


ax.scatter(
    hhi_figura[
        "hhi_frecuencia_normalizado"
    ],
    hhi_figura[
        "hhi_monetario_normalizado"
    ],
    s=55,
    alpha=0.65,
    color="#3D7EA6",
    edgecolor="white",
    linewidth=0.5
)


# ============================================================
# ============================================================
#
# Encima:
# concentración monetaria > frecuencia.
#
# Debajo:
# ============================================================

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.4,
    color="#666666"
)


# ============================================================
# ============================================================

top_5_diferencias = (
    hhi_figura
    .sort_values(
        "diferencia_absoluta",
        ascending=False
    )
    .head(5)
    .copy()
)


def dividir_nombre_figura_6(
    texto,
    ancho=28
):

    if pd.isna(texto):
        return "No informado"


    return "\n".join(
        textwrap.wrap(
            str(texto),
            width=ancho,
            break_long_words=False,
            break_on_hyphens=False
        )
    )


desplazamientos = [
    (16, 22),
    (16, -12),
    (16, 18),
    (16, -25),
    (16, 8)
]


for (
    (_, fila),
    desplazamiento
) in zip(
    top_5_diferencias.iterrows(),
    desplazamientos
):

    nombre = (
        dividir_nombre_figura_6(
            fila[
                "buyer_name"
            ]
        )
    )


    ax.annotate(
        nombre,

        (
            fila[
                "hhi_frecuencia_normalizado"
            ],

            fila[
                "hhi_monetario_normalizado"
            ]
        ),

        xytext=desplazamiento,

        textcoords="offset points",

        fontsize=8,

        ha="left",
        va="center",

        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="none",
            alpha=0.82
        ),

        arrowprops=dict(
            arrowstyle="-",
            linewidth=0.7,
            color="#777777"
        )
    )


# ============================================================
# ============================================================

ax.set_xlabel(
    "HHI normalizado por frecuencia",
    fontsize=11
)


ax.set_ylabel(
    "HHI normalizado por monto adjudicado",
    fontsize=11
)


ax.set_xlim(
    -0.02,
    1.02
)

ax.set_ylim(
    -0.02,
    1.02
)


ax.set_aspect(
    "equal",
    adjustable="box"
)


# ============================================================
# 26. ESTILO VISUAL
# ============================================================

ax.grid(
    True,
    linestyle="--",
    alpha=0.20
)

ax.set_axisbelow(True)


ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)


ax.tick_params(
    axis="both",
    labelsize=9.5
)


plt.tight_layout()


# ============================================================
# ============================================================

ruta_figura_6 = os.path.join(
    ruta_figuras,
    "figure_06_normalized_hhi_value_frequency_2025.png"
)


fig.savefig(
    ruta_figura_6,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    "\nFigura 6 guardada en:"
)

print(
    ruta_figura_6
)


plt.show()


In [ ]:
# ============================================================
# PASO 11. FIGURA 7
# ============================================================
#
# Objetivo:
# proveedores.
#
# BASE METODOLÓGICA:
#
#
#   puede ser identificada.
# - HHI normalizado:
#
#       HHI* = (HHI - 1/n) / (1 - 1/n)
#
#
#
# Indicadores complementarios:
#
# - CR4 monetario:
#
# - Entropía monetaria normalizada:
#   valores cercanos a 1 indican mayor diversidad;
#   valores cercanos a 0 indican menor diversidad.
#
# IMPORTANTE:
# ============================================================


import os
import textwrap
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


# ============================================================
# ============================================================
# ============================================================

columnas_requeridas_figura_7 = [
    "buyer_id",
    "buyer_name",
    "numero_adjudicaciones",
    "numero_proveedores",
    "hhi_monetario_normalizado",
    "cr4_monetario",
    "entropia_monetaria"
]


columnas_faltantes_figura_7 = [
    columna
    for columna in columnas_requeridas_figura_7
    if columna not in hhi_figura.columns
]


if columnas_faltantes_figura_7:

    raise ValueError(
        "Faltan columnas necesarias para la Figura 7: "
        + ", ".join(columnas_faltantes_figura_7)
        + ". Ejecutar primero el PASO 10 - FIGURA 6."
    )


# ============================================================
# ============================================================

base_figura_7 = hhi_figura[
    columnas_requeridas_figura_7
].copy()


# ------------------------------------------------------------
# ------------------------------------------------------------

base_figura_7 = (
    base_figura_7
    .dropna(
        subset=[
            "hhi_monetario_normalizado"
        ]
    )
    .copy()
)


print("=" * 75)
print("CONTROL DE FIGURA 7 - CONCENTRACIÓN MONETARIA")
print("=" * 75)

print(
    f"\nEntidades disponibles para el análisis: "
    f"{len(base_figura_7):,}"
)

print(
    f"Adjudicaciones mínimas entre las entidades analizadas: "
    f"{base_figura_7['numero_adjudicaciones'].min():,.0f}"
)

print(
    f"Número mínimo de proveedores: "
    f"{base_figura_7['numero_proveedores'].min():,.0f}"
)


# ============================================================
# ============================================================
# HHI monetario normalizado.
#
#
# ============================================================

top_concentracion_monetaria = (
    base_figura_7
    .sort_values(
        [
            "hhi_monetario_normalizado",
            "cr4_monetario"
        ],
        ascending=[
            False,
            False
        ]
    )
    .head(10)
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# ============================================================

top_concentracion_monetaria[
    "ranking"
] = (
    range(
        1,
        len(top_concentracion_monetaria) + 1
    )
)


# ============================================================
# ============================================================

tabla_figura_7 = (
    top_concentracion_monetaria[
        [
            "ranking",
            "buyer_name",
            "numero_adjudicaciones",
            "numero_proveedores",
            "hhi_monetario_normalizado",
            "cr4_monetario",
            "entropia_monetaria"
        ]
    ]
    .rename(
        columns={
            "ranking":
                "Ranking",

            "buyer_name":
                "Entidad contratante",

            "numero_adjudicaciones":
                "Adjudicaciones",

            "numero_proveedores":
                "Proveedores",

            "hhi_monetario_normalizado":
                "HHI* monetario",

            "cr4_monetario":
                "CR4 monetario",

            "entropia_monetaria":
                "Entropía monetaria"
        }
    )
    .copy()
)


# ------------------------------------------------------------
# ------------------------------------------------------------

tabla_figura_7[
    [
        "HHI* monetario",
        "CR4 monetario",
        "Entropía monetaria"
    ]
] = (
    tabla_figura_7[
        [
            "HHI* monetario",
            "CR4 monetario",
            "Entropía monetaria"
        ]
    ]
    .round(3)
)


print(
    "\n10 ENTIDADES CON MAYOR CONCENTRACIÓN MONETARIA\n"
)


display(
    tabla_figura_7
)


# ============================================================
# ============================================================

print(
    f"\nHHI* monetario máximo del Top 10: "
    f"{top_concentracion_monetaria['hhi_monetario_normalizado'].max():.3f}"
)

print(
    f"HHI* monetario mínimo del Top 10: "
    f"{top_concentracion_monetaria['hhi_monetario_normalizado'].min():.3f}"
)

print(
    f"CR4 monetario máximo del Top 10: "
    f"{top_concentracion_monetaria['cr4_monetario'].max():.3f}"
)

print(
    f"Entropía monetaria mínima del Top 10: "
    f"{top_concentracion_monetaria['entropia_monetaria'].min():.3f}"
)


# ============================================================
# ============================================================

def dividir_nombre_figura_7(
    texto,
    ancho=38
):

    if pd.isna(texto):
        return "No informado"

    return "\n".join(
        textwrap.wrap(
            str(texto),
            width=ancho,
            break_long_words=False,
            break_on_hyphens=False
        )
    )


top_concentracion_monetaria[
    "nombre_figura"
] = (
    top_concentracion_monetaria[
        "buyer_name"
    ]
    .apply(
        lambda texto:
        dividir_nombre_figura_7(
            texto,
            ancho=38
        )
    )
)


# ============================================================
# ============================================================

top_concentracion_grafico = (
    top_concentracion_monetaria
    .sort_values(
        "hhi_monetario_normalizado",
        ascending=True
    )
    .copy()
)


# ============================================================
# ============================================================

fig, ax = plt.subplots(
    figsize=(14, 10),
    dpi=150
)


barras = ax.barh(
    top_concentracion_grafico[
        "nombre_figura"
    ],

    top_concentracion_grafico[
        "hhi_monetario_normalizado"
    ],

    color="#4F7D95",
    height=0.62
)


# ============================================================
# ============================================================

for barra, valor in zip(

    barras,

    top_concentracion_grafico[
        "hhi_monetario_normalizado"
    ]
):

    ax.text(
        valor + 0.012,

        barra.get_y()
        + barra.get_height() / 2,

        f"{valor:.3f}",

        va="center",
        ha="left",

        fontsize=10.5,
        fontweight="bold"
    )


# ============================================================
# ============================================================

ax.set_xlabel(
    "HHI monetario normalizado",
    fontsize=11
)

ax.set_ylabel("")


ax.set_xlim(
    0,
    1.08
)


# ------------------------------------------------------------
# ------------------------------------------------------------

ax.set_xticks(
    [
        0.0,
        0.2,
        0.4,
        0.6,
        0.8,
        1.0
    ]
)


# ============================================================
# 12. ESTILO VISUAL
# ============================================================

ax.xaxis.grid(
    True,
    linestyle="--",
    alpha=0.20
)

ax.set_axisbelow(True)


ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

ax.spines[
    "left"
].set_visible(False)


ax.tick_params(
    axis="y",
    labelsize=9.5
)

ax.tick_params(
    axis="x",
    labelsize=10
)


# ============================================================
# ============================================================

plt.subplots_adjust(
    left=0.52,
    right=0.95,
    top=0.97,
    bottom=0.10
)


# ============================================================
# ============================================================

ruta_figura_7 = os.path.join(
    ruta_figuras,
    "figure_07_buyers_highest_value_concentration_2025.png"
)


fig.savefig(
    ruta_figura_7,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    "\nFigura 7 guardada en:"
)

print(
    ruta_figura_7
)


plt.show()


In [ ]:
# ============================================================
# PASO 12. FIGURA 8
# ============================================================
#
# Objetivo:
# proveedores.
#
# BASE METODOLÓGICA:
#
#
# - HHI normalizado:
#
#       HHI* = (HHI - 1/n) / (1 - 1/n)
#
#
#
# Indicadores complementarios:
#
#
#   valores cercanos a 1 indican mayor diversidad;
#   valores cercanos a 0 indican menor diversidad.
#
# IMPORTANTE:
# - Esta figura mide frecuencia, NO concentración monetaria.
# ============================================================


import os
import textwrap
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


# ============================================================
# ============================================================
# ============================================================

columnas_requeridas_figura_8 = [
    "buyer_id",
    "buyer_name",
    "numero_adjudicaciones",
    "numero_proveedores",
    "hhi_frecuencia_normalizado",
    "cr4_frecuencia",
    "entropia_frecuencia"
]


columnas_faltantes_figura_8 = [
    columna
    for columna in columnas_requeridas_figura_8
    if columna not in hhi_figura.columns
]


if columnas_faltantes_figura_8:

    raise ValueError(
        "Faltan columnas necesarias para la Figura 8: "
        + ", ".join(columnas_faltantes_figura_8)
        + ". Ejecutar primero el PASO 10 - FIGURA 6."
    )


# ============================================================
# ============================================================

base_figura_8 = hhi_figura[
    columnas_requeridas_figura_8
].copy()



base_figura_8 = (
    base_figura_8
    .dropna(
        subset=[
            "hhi_frecuencia_normalizado"
        ]
    )
    .copy()
)


# ============================================================
# 3. CONTROLES
# ============================================================

print("=" * 75)
print("CONTROL DE FIGURA 8 - CONCENTRACIÓN POR FRECUENCIA")
print("=" * 75)

print(
    f"\nEntidades disponibles para el análisis: "
    f"{len(base_figura_8):,}"
)

print(
    f"Adjudicaciones mínimas entre las entidades analizadas: "
    f"{base_figura_8['numero_adjudicaciones'].min():,.0f}"
)

print(
    f"Número mínimo de proveedores: "
    f"{base_figura_8['numero_proveedores'].min():,.0f}"
)


# ============================================================
# ============================================================
#
#
# ============================================================

top_concentracion_frecuencia = (
    base_figura_8
    .sort_values(
        [
            "hhi_frecuencia_normalizado",
            "cr4_frecuencia",
            "entropia_frecuencia"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .head(10)
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# ============================================================

top_concentracion_frecuencia[
    "ranking"
] = range(
    1,
    len(top_concentracion_frecuencia) + 1
)


# ============================================================
# ============================================================

tabla_figura_8 = (
    top_concentracion_frecuencia[
        [
            "ranking",
            "buyer_name",
            "numero_adjudicaciones",
            "numero_proveedores",
            "hhi_frecuencia_normalizado",
            "cr4_frecuencia",
            "entropia_frecuencia"
        ]
    ]
    .rename(
        columns={
            "ranking":
                "Ranking",

            "buyer_name":
                "Entidad contratante",

            "numero_adjudicaciones":
                "Adjudicaciones",

            "numero_proveedores":
                "Proveedores",

            "hhi_frecuencia_normalizado":
                "HHI* frecuencia",

            "cr4_frecuencia":
                "CR4 frecuencia",

            "entropia_frecuencia":
                "Entropía frecuencia"
        }
    )
    .copy()
)


# ------------------------------------------------------------
# ------------------------------------------------------------

tabla_figura_8[
    [
        "HHI* frecuencia",
        "CR4 frecuencia",
        "Entropía frecuencia"
    ]
] = (
    tabla_figura_8[
        [
            "HHI* frecuencia",
            "CR4 frecuencia",
            "Entropía frecuencia"
        ]
    ]
    .round(3)
)


print(
    "\n10 ENTIDADES CON MAYOR CONCENTRACIÓN POR FRECUENCIA\n"
)



display(
    tabla_figura_8
)


# ============================================================
# ============================================================

print(
    f"\nHHI* frecuencia máximo del Top 10: "
    f"{top_concentracion_frecuencia['hhi_frecuencia_normalizado'].max():.3f}"
)

print(
    f"HHI* frecuencia mínimo del Top 10: "
    f"{top_concentracion_frecuencia['hhi_frecuencia_normalizado'].min():.3f}"
)

print(
    f"CR4 frecuencia máximo del Top 10: "
    f"{top_concentracion_frecuencia['cr4_frecuencia'].max():.3f}"
)

print(
    f"Entropía frecuencia mínima del Top 10: "
    f"{top_concentracion_frecuencia['entropia_frecuencia'].min():.3f}"
)


# ============================================================
# ============================================================

def dividir_nombre_figura_8(
    texto,
    ancho=38
):

    if pd.isna(texto):
        return "No informado"

    return "\n".join(
        textwrap.wrap(
            str(texto),
            width=ancho,
            break_long_words=False,
            break_on_hyphens=False
        )
    )


top_concentracion_frecuencia[
    "nombre_figura"
] = (
    top_concentracion_frecuencia[
        "buyer_name"
    ]
    .apply(
        lambda texto:
        dividir_nombre_figura_8(
            texto,
            ancho=38
        )
    )
)


# ============================================================
# ============================================================

top_concentracion_frecuencia_grafico = (
    top_concentracion_frecuencia
    .sort_values(
        "hhi_frecuencia_normalizado",
        ascending=True
    )
    .copy()
)


# ============================================================
# ============================================================

fig, ax = plt.subplots(
    figsize=(14, 10),
    dpi=150
)


barras = ax.barh(
    top_concentracion_frecuencia_grafico[
        "nombre_figura"
    ],

    top_concentracion_frecuencia_grafico[
        "hhi_frecuencia_normalizado"
    ],

    color="#4F7D95",
    height=0.62
)


# ============================================================
# ============================================================

for barra, valor in zip(

    barras,

    top_concentracion_frecuencia_grafico[
        "hhi_frecuencia_normalizado"
    ]
):

    ax.text(
        valor + 0.012,

        barra.get_y()
        + barra.get_height() / 2,

        f"{valor:.3f}",

        va="center",
        ha="left",

        fontsize=10.5,
        fontweight="bold"
    )


# ============================================================
# ============================================================

ax.set_xlabel(
    "HHI normalizado por frecuencia",
    fontsize=11
)

ax.set_ylabel("")


# ------------------------------------------------------------
#
# Esto evita exagerar visualmente diferencias pequeñas
# ------------------------------------------------------------

ax.set_xlim(
    0,
    1.08
)


ax.set_xticks(
    [
        0.0,
        0.2,
        0.4,
        0.6,
        0.8,
        1.0
    ]
)


# ============================================================
# 13. ESTILO VISUAL
# ============================================================

ax.xaxis.grid(
    True,
    linestyle="--",
    alpha=0.20
)

ax.set_axisbelow(True)


ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

ax.spines[
    "left"
].set_visible(False)


ax.tick_params(
    axis="y",
    labelsize=9.5
)

ax.tick_params(
    axis="x",
    labelsize=10
)


# ============================================================
# 14. AJUSTAR MÁRGENES
# ============================================================

plt.subplots_adjust(
    left=0.52,
    right=0.95,
    top=0.97,
    bottom=0.10
)


# ============================================================
# ============================================================

ruta_figura_8 = os.path.join(
    ruta_figuras,
    "figure_08_buyers_highest_frequency_concentration_2025.png"
)


fig.savefig(
    ruta_figura_8,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    "\nFigura 8 guardada en:"
)

print(
    ruta_figura_8
)


plt.show()
